# Independent Dataset Profiling and Acceptance Review
**Medicine Expiry Management System: FEFO-based synthetic dataset (ten CSV files)**

## Purpose
This notebook independently verifies the synthetic dataset built for the *Web-Based Demand-Driven Medicine Expiry Management System* **before any of it is imported into the Django database**. It does not rely on the generator's built-in validation. Every rule is re-tested from the exported CSV files alone, and every test is logged so that the final acceptance report is computed from the results instead of being typed in.

## The data process
| Stage | What was done | Evidence in the package |
|---|---|---|
| 1. Define the rules | Period, volumes, FEFO rules, receiving rules, forecasting, procurement and integrity rules were written down before generation. | `documentation/generation_methodology.md` |
| 2. Align with the application | The generator was designed against the Django models and business logic (fields, statuses, FEFO eligibility, procurement formula). | `documentation/django_schema_audit.txt`, `django_field_mapping.md`, `business_logic_audit.txt`, `final_business_logic_audit.txt` |
| 3. Generate | A fixed-seed script (seed 20261109) produced the ten CSV files. There was no Django connection and no database import. | `scripts/generation_script.py`, `reports/generation_summary.txt` and `.json` |
| 4. Validate while generating | `validate_generated_data()` checks keys, stock, the FEFO replay, forecasts, procurement and receiving before the files are exported. | `scripts/generation_script.py` (section 12) |
| 5. **Independent profiling** | **This notebook** re-tests the exported files: structure, completeness, keys, dates, quantities, FEFO, overrides, suppliers and reconciliations. | This notebook, `reports/profiling/` |
| 6. Decision | Accept with documented limitations. Import into Django is **not yet approved**. | Section 9 |

## About data cleaning
The methodology planned for the data to be *profiled, cleaned and validated*. This notebook carries out the profiling and validation. **It found nothing that required the data to be changed**, so no record is modified, removed or filled in, and no CSV file is rewritten. The only handling happens at read time (for example, reading phone numbers as text). Section 10 logs every finding, the decision taken and the action, so the whole process can be traced.

## How each section works
1. **Purpose and method**: what is being tested and how.
2. **Code**: the test itself. Each test is recorded with `record(...)` as *observed vs expected*.
3. **Confirmation table**: the checks of that section with `PASS` or `FAIL`.
4. **Findings**: what the results mean for importing the data.

## Contents
| Section | Topic |
|---|---|
| 0 | Setup, dataset inventory and loading |
| 1 | Structure and completeness |
| 2 | Primary keys and foreign keys |
| 3 | Dates and quantities |
| 4 | FEFO compliance |
| 5 | Dispensing overrides |
| 6 | Supplier master data |
| 7 | Procurement and receiving reconciliation |
| 8 | Forecasts, recommendations and purchase orders |
| 9 | Acceptance report |
| 10 | Findings and decisions log |
| A | Appendix: generator rules referred to in this review |

**How to run:** keep this notebook inside `synthetic_dataset_package/notebooks/` (any folder below the project root also works), then choose *Run All*. The notebook only reads the datasets. The one file it writes is `reports/profiling/final_dataset_acceptance_report.txt`.

---
## 0. Setup, dataset inventory and loading
**Purpose:** prepare the environment, locate the ten files and confirm that each loads with the expected number of records.

**Method:** the next cell loads the libraries, finds the project folder, and defines two helpers used throughout:
- `record(...)` logs one check as *observed vs expected* (a violation count is expected to be 0).
- `section_table(...)` shows the confirmations of one section.

In [2]:
# 0.1  Libraries, project paths, constants and the check registry
import sys
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

print("Independent Dataset Profiling: Medicine Expiry Management System")
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


def find_project_root(start=None):
    """Walk up from the working folder until the dataset package is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "synthetic_dataset_package" / "datasets").is_dir():
            return folder
    raise FileNotFoundError(
        f"synthetic_dataset_package/datasets not found above {start}"
    )


PROJECT_ROOT = find_project_root()
PACKAGE_DIR = PROJECT_ROOT / "synthetic_dataset_package"
DATASET_DIR = PACKAGE_DIR / "datasets"
REPORT_DIR = PACKAGE_DIR / "reports" / "profiling"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("\nDataset folder:", DATASET_DIR.relative_to(PROJECT_ROOT))
print("Report folder :", REPORT_DIR.relative_to(PROJECT_ROOT))

# Reference values taken from the methodology and the generator
SNAPSHOT_DATE = pd.Timestamp("2026-11-09")   # inventory snapshot
MAX_PRESCRIPTION_DAYS = 14                   # longest prescription period

# Check registry: every verification in the notebook is logged here
RESULTS = []   # one row per check
LIMITS = {}    # counts that feed the documented limitations


def record(section, check, observed, expected=0, group=None, key=None):
    """Log one check. It passes when observed == expected."""
    observed, expected = int(observed), int(expected)
    RESULTS.append({
        "Section": section,
        "Check": check,
        "Observed": observed,
        "Expected": expected,
        "Status": "PASS" if observed == expected else "FAIL",
        "Group": group,
        "Key": key,
    })


def section_table(section):
    """Return the confirmation table for one section."""
    rows = [row for row in RESULTS if row["Section"] == section]
    table = pd.DataFrame(rows)[["Check", "Observed", "Expected", "Status"]]
    passed = int((table["Status"] == "PASS").sum())
    print(f"\nSECTION {section} CONFIRMATION: {passed}/{len(table)} checks passed")
    return table

Independent Dataset Profiling: Medicine Expiry Management System
Python : 3.14.6
Pandas : 3.1.1
NumPy  : 2.5.4

Dataset folder: synthetic_dataset_package\datasets
Report folder : synthetic_dataset_package\reports\profiling


### The ten datasets
| Dataset | File | Primary key | Content |
|---|---|---|---|
| `medicines` | `medicines.csv` | `medicine_id` | 100 fictional medicines with reorder level and safety-stock days |
| `suppliers` | `suppliers.csv` | `supplier_id` | 15 fictional suppliers with contact details and delivery lead time |
| `medicine_batches` | `medicine_batches.csv` | `batch_id` | 2,500 batches with received and expiry dates, quantities and status |
| `dispensing_transactions` | `dispensing_transactions.csv` | `transaction_id` | 42,617 dispensing rows, each referring to one batch |
| `demand_forecasts_evaluation` | `demand_forecasts_evaluation.csv` | `forecast_id` | 600 monthly forecasts (May to October 2026) compared with actual demand |
| `demand_forecasts_november` | `demand_forecasts_november_2026.csv` | `forecast_id` | 100 forward forecasts for November 2026 |
| `procurement_recommendations` | `procurement_recommendations.csv` | `recommendation_id` | 2,500 recommendations (2,400 historical and 100 for November) |
| `purchase_orders` | `purchase_orders.csv` | `purchase_order_id` | 2,455 purchase orders |
| `goods_receipts` | `goods_receipts.csv` | `goods_receipt_id` | 2,400 goods receipts |
| `goods_receipt_items` | `goods_receipt_items.csv` | `receipt_item_id` | 2,400 receipt items, each creating one batch |

**Reference periods (from the methodology):** history 1 November 2024 to 31 October 2026 (training to 30 April 2026, evaluation 1 May to 31 October 2026), then 1 to 9 November 2026 up to the inventory snapshot (snapshot-period dispensing), and a forward forecast for November 2026.

**Loading rule:** the files are read exactly as stored (`utf-8-sig` removes a byte-order mark if one exists) and are never written back.

In [3]:
# 0.2  Locate and load the ten datasets; confirm record counts
dataset_files = {
    "medicines": "medicines.csv",
    "suppliers": "suppliers.csv",
    "medicine_batches": "medicine_batches.csv",
    "dispensing_transactions": "dispensing_transactions.csv",
    "demand_forecasts_evaluation": "demand_forecasts_evaluation.csv",
    "demand_forecasts_november": "demand_forecasts_november_2026.csv",
    "procurement_recommendations": "procurement_recommendations.csv",
    "purchase_orders": "purchase_orders.csv",
    "goods_receipts": "goods_receipts.csv",
    "goods_receipt_items": "goods_receipt_items.csv",
}

primary_keys = {
    "medicines": "medicine_id",
    "suppliers": "supplier_id",
    "medicine_batches": "batch_id",
    "dispensing_transactions": "transaction_id",
    "demand_forecasts_evaluation": "forecast_id",
    "demand_forecasts_november": "forecast_id",
    "procurement_recommendations": "recommendation_id",
    "purchase_orders": "purchase_order_id",
    "goods_receipts": "goods_receipt_id",
    "goods_receipt_items": "receipt_item_id",
}

expected_counts = {
    "medicines": 100,
    "suppliers": 15,
    "medicine_batches": 2500,
    "dispensing_transactions": 42617,
    "demand_forecasts_evaluation": 600,
    "demand_forecasts_november": 100,
    "procurement_recommendations": 2500,
    "purchase_orders": 2455,
    "goods_receipts": 2400,
    "goods_receipt_items": 2400,
}

for filename in dataset_files.values():
    assert (DATASET_DIR / filename).is_file(), f"Dataset file missing: {filename}"
print("All ten dataset files located.\n")

datasets = {}
for name, filename in dataset_files.items():
    datasets[name] = pd.read_csv(
        DATASET_DIR / filename, encoding="utf-8-sig", low_memory=False
    )
    record("0", f"{name}: record count", len(datasets[name]),
           expected_counts[name], group="Record counts")
    print(f"{name:<32} {len(datasets[name]):>7,} records")

display(section_table("0"))

All ten dataset files located.

medicines                            100 records
suppliers                             15 records
medicine_batches                   2,500 records
dispensing_transactions           42,617 records
demand_forecasts_evaluation          600 records
demand_forecasts_november            100 records
procurement_recommendations        2,500 records
purchase_orders                    2,455 records
goods_receipts                     2,400 records
goods_receipt_items                2,400 records

SECTION 0 CONFIRMATION: 10/10 checks passed


,Check,Observed,Expected,Status
0,medicines: record count,100,100,PASS
1,suppliers: record count,15,15,PASS
2,medicine_batches: record count,2500,2500,PASS
3,dispensing_transactions: record count,42617,42617,PASS
4,demand_forecasts_evaluation: record count,600,600,PASS
5,demand_forecasts_november: record count,100,100,PASS
6,procurement_recommendations: record count,2500,2500,PASS
7,purchase_orders: record count,2455,2455,PASS
8,goods_receipts: record count,2400,2400,PASS
9,goods_receipt_items: record count,2400,2400,PASS


---
## 1. Structure and completeness
**Purpose:** confirm that every file has a readable schema and find out where values are missing, before the relationships and business rules are tested.

**Method:**
1. *Structure:* list the rows, columns, data types and non-null counts of every dataset.
2. *Missing values:* count genuine nulls and empty text fields (`""` after trimming spaces) in every column.
3. *Forecast link:* the structure shows that `procurement_recommendations.forecast_id` is mostly empty, so test whether this follows the generator's design or points to a broken relationship.

**Pass criterion:** every missing value must be explained by the design. Any column with missing values outside the three expected ones is a failure.

In [4]:
# 1.1  Structural profile of every dataset
for dataset_name, df in datasets.items():
    print("\n" + "=" * 85)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 85)
    print(f"Rows: {df.shape[0]:,}    Columns: {df.shape[1]}")

    structure = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.notna().sum().values,
    })
    display(structure)


DATASET: MEDICINES
Rows: 100    Columns: 8


,Column,Data Type,Non-Null Count
0,medicine_id,str,100
1,medicine_name,str,100
2,generic_name,str,100
3,category,str,100
4,dosage_form,str,100
5,reorder_level,int64,100
6,safety_stock_days,int64,100
7,is_active,bool,100



DATASET: SUPPLIERS
Rows: 15    Columns: 8


,Column,Data Type,Non-Null Count
0,supplier_id,str,15
1,supplier_name,str,15
2,contact_person,str,15
3,email,str,15
4,phone_number,int64,15
5,address,str,15
6,delivery_lead_time_days,int64,15
7,is_active,bool,15



DATASET: MEDICINE_BATCHES
Rows: 2,500    Columns: 9


,Column,Data Type,Non-Null Count
0,batch_id,str,2500
1,medicine_id,str,2500
2,batch_number,str,2500
3,quantity_received,int64,2500
4,quantity_available,int64,2500
5,received_date,str,2500
6,expiry_date,str,2500
7,storage_location,str,2500
8,status,str,2500


... (10 outputs left out)


DATASET: GOODS_RECEIPTS
Rows: 2,400    Columns: 6


,Column,Data Type,Non-Null Count
0,goods_receipt_id,str,2400
1,purchase_order_id,str,2400
2,delivery_date,str,2400
3,received_at,str,2400
4,delivery_reference,str,2400
5,notes,str,2400



DATASET: GOODS_RECEIPT_ITEMS
Rows: 2,400    Columns: 10


,Column,Data Type,Non-Null Count
0,receipt_item_id,str,2400
1,goods_receipt_id,str,2400
2,medicine_batch_id,str,2400
3,batch_number,str,2400
4,quantity_received,int64,2400
5,expiry_date,str,2400
6,storage_location,str,2400
7,inspection_status,str,2400
8,inspected_at,str,2400
9,inspection_notes,str,2400


In [5]:
# 1.2  Missing-value and blank-field audit (all ten datasets)
missing_records = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        null_count = int(df[column].isna().sum())

        blank_count = 0
        if df[column].dtype == "object" or pd.api.types.is_string_dtype(df[column]):
            blank_count = int(
                df[column].astype("string").str.strip().eq("").fillna(False).sum()
            )

        missing_records.append({
            "Dataset": dataset_name,
            "Column": column,
            "Null Values": null_count,
            "Blank Values": blank_count,
            "Total Missing": null_count + blank_count,
        })

missing_report = pd.DataFrame(missing_records)
issues = (
    missing_report[missing_report["Total Missing"] > 0]
    .sort_values(["Dataset", "Total Missing"], ascending=[True, False])
    .reset_index(drop=True)
)

print("MISSING-VALUE AND BLANK-FIELD AUDIT")
print("=" * 85)
print(f"Columns checked: {len(missing_report)}")

if issues.empty:
    print("No missing or blank values found.")
else:
    display(issues)

# The only gaps the design allows (explained in the findings below)
expected_gaps = {
    ("demand_forecasts_november", "actual_demand_quantity"),
    ("demand_forecasts_november", "absolute_error"),
    ("procurement_recommendations", "forecast_id"),
}
found_gaps = set(zip(issues["Dataset"], issues["Column"]))

record("1", "Columns with missing or blank values outside the expected three",
       len(found_gaps - expected_gaps), 0, group="Completeness")
record("1", "Expected gap columns that were not found",
       len(expected_gaps - found_gaps), 0, group="Completeness")

MISSING-VALUE AND BLANK-FIELD AUDIT
Columns checked: 103


,Dataset,Column,Null Values,Blank Values,Total Missing
0,demand_forecasts_november,actual_demand_quantity,100,0,100
1,demand_forecasts_november,absolute_error,100,0,100
2,procurement_recommendations,forecast_id,2400,0,2400


In [6]:
# 1.3  Why 2,400 recommendations have no forecast_id
recommendations = datasets["procurement_recommendations"]
november_forecasts = datasets["demand_forecasts_november"]
evaluation_forecasts = datasets["demand_forecasts_evaluation"]

known_forecast_ids = set(
    pd.concat([
        november_forecasts["forecast_id"],
        evaluation_forecasts["forecast_id"],
    ]).dropna().astype(str)
)

linked = recommendations[recommendations["forecast_id"].notna()]
unlinked = recommendations[recommendations["forecast_id"].isna()]
linked_ids = set(linked["forecast_id"].astype(str))

print("PROCUREMENT–FORECAST RELATIONSHIP AUDIT")
print("=" * 75)
print("Total recommendations:", len(recommendations))
print("Recommendations with forecast IDs:", len(linked))
print("Recommendations without forecast IDs:", len(unlinked))
print("November forecasts:", len(november_forecasts))
print("Evaluation forecasts:", len(evaluation_forecasts))
print("Unique linked forecast IDs:", len(linked_ids))
print("All populated forecast IDs exist:", linked_ids.issubset(known_forecast_ids))

created = pd.to_datetime(recommendations["created_at"], errors="coerce")

print("\nRecommendations by creation month:")
print(
    recommendations.assign(creation_month=created.dt.to_period("M"))
    .groupby("creation_month", dropna=False)
    .agg(total_recommendations=("recommendation_id", "count"),
         linked_forecasts=("forecast_id", "count"))
    .to_string()
)

print("\nSample linked recommendations:")
display(linked[["recommendation_id", "medicine_id", "forecast_id", "created_at"]].head(5))
print("Sample unlinked recommendations:")
display(unlinked[["recommendation_id", "medicine_id", "forecast_id", "created_at"]].head(5))

is_november = created >= pd.Timestamp("2026-11-01")
record("1", "Populated forecast IDs that do not exist in the forecast files",
       len(linked_ids - known_forecast_ids), 0, group="Completeness")
record("1", "November recommendations without a forecast_id",
       (is_november & recommendations["forecast_id"].isna()).sum(), 0, group="Completeness")
record("1", "Historical recommendations that carry a forecast_id",
       (~is_november & recommendations["forecast_id"].notna()).sum(), 0, group="Completeness")

LIMITS["no_forecast_id"] = len(unlinked)
display(section_table("1"))

PROCUREMENT–FORECAST RELATIONSHIP AUDIT
Total recommendations: 2500
Recommendations with forecast IDs: 100
Recommendations without forecast IDs: 2400
November forecasts: 100
Evaluation forecasts: 600
Unique linked forecast IDs: 100
All populated forecast IDs exist: True

Recommendations by creation month:
                total_recommendations  linked_forecasts
creation_month                                         
2024-10                           100                 0
2024-11                           100                 0
2024-12                           100                 0
2025-01                           100                 0
2025-02                           100                 0
2025-03                           100                 0
2025-04                           100                 0
2025-05                           100                 0
2025-06                           100                 0
2025-07                           100                 0
2025-08              

,recommendation_id,medicine_id,forecast_id,created_at
2400,REC002401,MED001,FCN000601,2026-11-09T09:00:00
2401,REC002402,MED002,FCN000602,2026-11-09T09:00:00
2402,REC002403,MED003,FCN000603,2026-11-09T09:00:00
2403,REC002404,MED004,FCN000604,2026-11-09T09:00:00
2404,REC002405,MED005,FCN000605,2026-11-09T09:00:00


Sample unlinked recommendations:


,recommendation_id,medicine_id,forecast_id,created_at
0,REC000001,MED001,NaN,2024-10-25T09:00:00
1,REC000002,MED001,NaN,2024-11-24T09:00:00
2,REC000003,MED001,NaN,2024-12-25T09:00:00
3,REC000004,MED001,NaN,2025-01-25T09:00:00
4,REC000005,MED001,NaN,2025-02-22T09:00:00



SECTION 1 CONFIRMATION: 5/5 checks passed


,Check,Observed,Expected,Status
0,Columns with missing or blank values outside t...,0,0,PASS
1,Expected gap columns that were not found,0,0,PASS
2,Populated forecast IDs that do not exist in th...,0,0,PASS
3,November recommendations without a forecast_id,0,0,PASS
4,Historical recommendations that carry a foreca...,0,0,PASS


### Findings: structure and completeness
- All ten datasets have readable schemas and the expected record counts.
- Exactly **three columns** contain missing values, and all three are explained by the design:
  1. `demand_forecasts_november.actual_demand_quantity` and `absolute_error` are empty for all 100 rows. November has not happened yet, so the forecasts cannot be evaluated.
  2. `procurement_recommendations.forecast_id` is empty for **2,400 historical recommendations**. These were created monthly from October 2024 to September 2026 (24 batches of 100) and are not tied to a forecast. The 100 November recommendations, created on the 9 November snapshot, all carry a `forecast_id`, and every populated ID exists in the forecast files. The application model allows this gap, because `ProcurementRecommendation.forecast` is nullable.
- **Data-type observation:** `suppliers.phone_number` loads as an integer (`int64`) because pandas reads `+2547…` as a number. It is stored as text in the file. It is read as text in Section 6 and must also be imported as text. This is logged in Section 10.

**Result:** accepted. The 2,400 empty `forecast_id` values are recorded as a documented limitation.

---
## 2. Primary keys and foreign keys
**Purpose:** confirm that every record can be identified uniquely and that every reference points to a record that exists. Duplicate keys or orphaned references would break relationships or fail the database import.

**Method:**
1. *Primary keys:* for each dataset count duplicate and null IDs and compare the number of unique IDs with the number of rows.
2. *Foreign keys:* for each of the 13 relationships, check that every populated child value exists in the parent dataset. Forecast references are checked against the evaluation and November forecasts combined. Empty `forecast_id` values were explained in Section 1 and are excluded.

**Pass criterion:** zero duplicate IDs, zero null IDs and zero unmatched references.

In [7]:
# 2.1  Primary-key uniqueness
duplicate_results = []

for dataset_name, primary_key in primary_keys.items():
    df = datasets[dataset_name]
    duplicate_results.append({
        "Dataset": dataset_name,
        "Primary Key": primary_key,
        "Duplicate IDs": int(df[primary_key].duplicated().sum()),
        "Null IDs": int(df[primary_key].isna().sum()),
        "Unique IDs": int(df[primary_key].nunique()),
        "Rows": len(df),
    })

primary_key_report = pd.DataFrame(duplicate_results)
display(primary_key_report)

print("PRIMARY-KEY AUDIT SUMMARY")
print("=" * 65)
print("Datasets with duplicate IDs:", int((primary_key_report["Duplicate IDs"] > 0).sum()))
print("Datasets with null IDs:", int((primary_key_report["Null IDs"] > 0).sum()))

record("2", "Duplicate primary keys (all ten datasets)",
       primary_key_report["Duplicate IDs"].sum(), 0, group="Primary keys")
record("2", "Null primary keys (all ten datasets)",
       primary_key_report["Null IDs"].sum(), 0, group="Primary keys")
record("2", "Datasets whose unique ID count equals the row count",
       (primary_key_report["Unique IDs"] == primary_key_report["Rows"]).sum(),
       len(primary_key_report), group="Primary keys")

# 2.2  Foreign-key integrity (13 relationships)
all_forecasts = pd.concat(
    [datasets["demand_forecasts_evaluation"], datasets["demand_forecasts_november"]],
    ignore_index=True,
)
parent_datasets = {**datasets, "combined_forecasts": all_forecasts}

foreign_key_checks = [
    ("medicine_batches", "medicine_id", "medicines", "medicine_id"),
    ("dispensing_transactions", "medicine_id", "medicines", "medicine_id"),
    ("dispensing_transactions", "batch_id", "medicine_batches", "batch_id"),
    ("demand_forecasts_evaluation", "medicine_id", "medicines", "medicine_id"),
    ("demand_forecasts_november", "medicine_id", "medicines", "medicine_id"),
    ("procurement_recommendations", "medicine_id", "medicines", "medicine_id"),
    ("procurement_recommendations", "forecast_id", "combined_forecasts", "forecast_id"),
    ("purchase_orders", "medicine_id", "medicines", "medicine_id"),
    ("purchase_orders", "supplier_id", "suppliers", "supplier_id"),
    ("purchase_orders", "recommendation_id", "procurement_recommendations", "recommendation_id"),
    ("goods_receipts", "purchase_order_id", "purchase_orders", "purchase_order_id"),
    ("goods_receipt_items", "goods_receipt_id", "goods_receipts", "goods_receipt_id"),
    ("goods_receipt_items", "medicine_batch_id", "medicine_batches", "batch_id"),
]

foreign_key_results = []
for child_name, child_column, parent_name, parent_column in foreign_key_checks:
    child_values = datasets[child_name][child_column].dropna().astype(str)
    parent_values = set(parent_datasets[parent_name][parent_column].dropna().astype(str))
    unmatched = child_values[~child_values.isin(parent_values)]

    foreign_key_results.append({
        "Child Dataset": child_name,
        "Foreign Key": child_column,
        "Parent Dataset": parent_name,
        "Checked Records": len(child_values),
        "Unmatched References": len(unmatched),
        "Result": "PASS" if unmatched.empty else "FAIL",
    })

foreign_key_report = pd.DataFrame(foreign_key_results)
display(foreign_key_report)

print("FOREIGN-KEY AUDIT SUMMARY")
print("=" * 65)
print("Relationships checked:", len(foreign_key_report))
print("Failed relationships:", int((foreign_key_report["Result"] == "FAIL").sum()))
print("Total unmatched references:", int(foreign_key_report["Unmatched References"].sum()))

record("2", "Foreign-key relationships checked", len(foreign_key_report), 13,
       group="Foreign keys")
record("2", "Unmatched foreign-key references (13 relationships)",
       foreign_key_report["Unmatched References"].sum(), 0,
       group="Foreign keys", key="Foreign-key violations")

display(section_table("2"))

,Dataset,Primary Key,Duplicate IDs,Null IDs,Unique IDs,Rows
0,medicines,medicine_id,0,0,100,100
1,suppliers,supplier_id,0,0,15,15
2,medicine_batches,batch_id,0,0,2500,2500
3,dispensing_transactions,transaction_id,0,0,42617,42617
4,demand_forecasts_evaluation,forecast_id,0,0,600,600
5,demand_forecasts_november,forecast_id,0,0,100,100
6,procurement_recommendations,recommendation_id,0,0,2500,2500
7,purchase_orders,purchase_order_id,0,0,2455,2455
8,goods_receipts,goods_receipt_id,0,0,2400,2400
9,goods_receipt_items,receipt_item_id,0,0,2400,2400


PRIMARY-KEY AUDIT SUMMARY
Datasets with duplicate IDs: 0
Datasets with null IDs: 0


,Child Dataset,Foreign Key,Parent Dataset,Checked Records,Unmatched References,Result
0,medicine_batches,medicine_id,medicines,2500,0,PASS
1,dispensing_transactions,medicine_id,medicines,42617,0,PASS
2,dispensing_transactions,batch_id,medicine_batches,42617,0,PASS
3,demand_forecasts_evaluation,medicine_id,medicines,600,0,PASS
4,demand_forecasts_november,medicine_id,medicines,100,0,PASS
5,procurement_recommendations,medicine_id,medicines,2500,0,PASS
6,procurement_recommendations,forecast_id,combined_forecasts,100,0,PASS
7,purchase_orders,medicine_id,medicines,2455,0,PASS
8,purchase_orders,supplier_id,suppliers,2455,0,PASS
9,purchase_orders,recommendation_id,procurement_recommendations,2455,0,PASS


FOREIGN-KEY AUDIT SUMMARY
Relationships checked: 13
Failed relationships: 0
Total unmatched references: 0

SECTION 2 CONFIRMATION: 5/5 checks passed


,Check,Observed,Expected,Status
0,Duplicate primary keys (all ten datasets),0,0,PASS
1,Null primary keys (all ten datasets),0,0,PASS
2,Datasets whose unique ID count equals the row ...,10,10,PASS
3,Foreign-key relationships checked,13,13,PASS
4,Unmatched foreign-key references (13 relations...,0,0,PASS


### Findings: keys and relationships
- Every dataset has **unique, non-null primary keys**: the unique ID count equals the row count in all ten datasets (100, 15, 2,500, 42,617, 600, 100, 2,500, 2,455, 2,400 and 2,400).
- All **13 foreign-key relationships pass** with zero unmatched references, including the 100 populated forecast references. There are no orphaned records.
- The data can therefore be imported in parent-to-child order without referential errors.

---
## 3. Dates and quantities
**Purpose:** confirm that date fields hold real dates, that events happen in a possible order, and that quantities are valid. Valid formats alone do not prove the data is sensible, so the second step tests sequence and the third tests numbers.

**Method:**
1. *Date validity:* parse all 21 date fields (into copies, never changing the data) and count values that cannot be parsed. The earliest and latest dates show the covered period.
2. *Chronology:* seven sequence rules, for example no dispensing before the batch was received or after it expired.
3. *Quantities:* seven rules, for example no negative stock and no batch with more available stock than was received.

**Pass criterion:** zero invalid dates and zero violations in all 14 rules.

In [8]:
# 3.1  Date-field validity (parsing is done on copies; the data is not modified)
date_columns = {
    "medicine_batches": ["received_date", "expiry_date"],
    "dispensing_transactions": ["dispensed_date", "dispensed_at"],
    "demand_forecasts_evaluation": ["forecast_period_start", "forecast_period_end",
                                    "history_start_date", "history_end_date", "generated_at"],
    "demand_forecasts_november": ["forecast_period_start", "forecast_period_end",
                                  "history_start_date", "history_end_date", "generated_at"],
    "procurement_recommendations": ["created_at", "updated_at"],
    "purchase_orders": ["order_date"],
    "goods_receipts": ["delivery_date", "received_at"],
    "goods_receipt_items": ["expiry_date", "inspected_at"],
}

date_results = []
for dataset_name, columns in date_columns.items():
    df = datasets[dataset_name]
    for column in columns:
        parsed = pd.to_datetime(df[column], format="mixed", errors="coerce")
        date_results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Invalid Dates": int(parsed.isna().sum()),
            "Earliest Date": parsed.min(),
            "Latest Date": parsed.max(),
        })

date_report = pd.DataFrame(date_results)
display(date_report)

print("DATE-FIELD AUDIT SUMMARY")
print("=" * 65)
print("Date fields checked:", len(date_report))
print("Fields containing invalid dates:", int((date_report["Invalid Dates"] > 0).sum()))
print("Total invalid date values:", int(date_report["Invalid Dates"].sum()))

# Dispensing runs up to the snapshot (1–9 Nov 2026 is the snapshot period) and not beyond it
dispensed_on = pd.to_datetime(datasets["dispensing_transactions"]["dispensed_at"])
snapshot_period_rows = int((dispensed_on >= pd.Timestamp("2026-11-01")).sum())
dispensed_after_snapshot = int((dispensed_on >= SNAPSHOT_DATE + pd.Timedelta(days=1)).sum())
print("Transactions dated 1–9 November 2026 (snapshot period):", snapshot_period_rows)
print("Transactions dated after the snapshot date:", dispensed_after_snapshot)

record("3", "Date fields checked", len(date_report), 21, group="Date validity")
record("3", "Invalid date values", date_report["Invalid Dates"].sum(), 0, group="Date validity")
record("3", "Dispensing transactions dated after the inventory snapshot",
       dispensed_after_snapshot, 0, group="Date validity")

,Dataset,Column,Invalid Dates,Earliest Date,Latest Date
0,medicine_batches,received_date,0,2024-10-15 00:00:00,2026-10-01 00:00:00
1,medicine_batches,expiry_date,0,2026-04-25 00:00:00,2029-03-19 00:00:00
2,dispensing_transactions,dispensed_date,0,2024-11-01 08:03:00,2026-11-09 16:54:00
3,dispensing_transactions,dispensed_at,0,2024-11-01 08:03:00,2026-11-09 16:54:00
4,demand_forecasts_evaluation,forecast_period_start,0,2026-05-01 00:00:00,2026-10-01 00:00:00
5,demand_forecasts_evaluation,forecast_period_end,0,2026-05-31 00:00:00,2026-10-31 00:00:00
6,demand_forecasts_evaluation,history_start_date,0,2026-01-31 00:00:00,2026-07-03 00:00:00
7,demand_forecasts_evaluation,history_end_date,0,2026-05-01 00:00:00,2026-10-01 00:00:00
8,demand_forecasts_evaluation,generated_at,0,2026-05-01 08:00:00,2026-10-01 08:00:00
9,demand_forecasts_november,forecast_period_start,0,2026-11-01 00:00:00,2026-11-01 00:00:00


DATE-FIELD AUDIT SUMMARY
Date fields checked: 21
Fields containing invalid dates: 0
Total invalid date values: 0
Transactions dated 1–9 November 2026 (snapshot period): 551
Transactions dated after the snapshot date: 0


In [9]:
# 3.2  Chronological consistency (7 rules) and quantity integrity (7 rules)
batches = datasets["medicine_batches"].copy()
transactions = datasets["dispensing_transactions"].copy()
receipts = datasets["goods_receipts"].copy()
receipt_items = datasets["goods_receipt_items"].copy()
suppliers = datasets["suppliers"]
medicines = datasets["medicines"]

# Convert date fields on the copies
for df, columns in [
    (batches, ["received_date", "expiry_date"]),
    (transactions, ["dispensed_at"]),
    (receipts, ["delivery_date", "received_at"]),
    (receipt_items, ["expiry_date", "inspected_at"]),
]:
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

# Relationships needed for the sequence rules
dispensing_check = transactions.merge(
    batches[["batch_id", "received_date", "expiry_date"]], on="batch_id", how="left"
)
receipt_check = receipt_items.merge(
    receipts[["goods_receipt_id", "delivery_date", "received_at"]],
    on="goods_receipt_id", how="left",
)
batch_reference = batches[["batch_id", "received_date", "expiry_date"]].rename(columns={
    "batch_id": "medicine_batch_id",
    "received_date": "batch_received_date",
    "expiry_date": "batch_expiry_date",
})
batch_receipt_check = receipt_items.merge(
    batch_reference, on="medicine_batch_id", how="left"
).merge(
    receipts[["goods_receipt_id", "delivery_date"]], on="goods_receipt_id", how="left"
)

chronology_results = [
    {"Check": "Batch expiry after receipt",
     "Violations": int((batches["expiry_date"] < batches["received_date"]).sum())},
    {"Check": "Dispensing before batch receipt",
     "Violations": int((dispensing_check["dispensed_at"] < dispensing_check["received_date"]).sum())},
    {"Check": "Dispensing after batch expiry",
     "Violations": int((dispensing_check["dispensed_at"].dt.normalize()
                        > dispensing_check["expiry_date"]).sum())},
    {"Check": "Goods receipt before delivery date",
     "Violations": int((receipts["received_at"].dt.normalize() < receipts["delivery_date"]).sum())},
    {"Check": "Inspection before physical receipt",
     "Violations": int((receipt_check["inspected_at"] < receipt_check["received_at"]).sum())},
    {"Check": "Receipt-item expiry differs from batch expiry",
     "Violations": int((batch_receipt_check["expiry_date"]
                        != batch_receipt_check["batch_expiry_date"]).sum())},
    {"Check": "Batch receipt date differs from goods receipt date",
     "Violations": int((batch_receipt_check["batch_received_date"]
                        != batch_receipt_check["delivery_date"]).sum())},
]
chronology_report = pd.DataFrame(chronology_results)

print("CHRONOLOGICAL CONSISTENCY")
print("=" * 65)
display(chronology_report)
print("Checks performed:", len(chronology_report))
print("Total violations:", int(chronology_report["Violations"].sum()))

for row in chronology_results:
    record("3", row["Check"], row["Violations"], 0, group="Chronology")

quantity_results = [
    {"Check": "Batch quantity received <= 0",
     "Violations": int((batches["quantity_received"] <= 0).sum())},
    {"Check": "Batch quantity available < 0",
     "Violations": int((batches["quantity_available"] < 0).sum())},
    {"Check": "Available quantity exceeds received quantity",
     "Violations": int((batches["quantity_available"] > batches["quantity_received"]).sum())},
    {"Check": "Dispensed quantity <= 0",
     "Violations": int((transactions["quantity_dispensed"] <= 0).sum())},
    {"Check": "Supplier delivery lead time <= 0",
     "Violations": int((suppliers["delivery_lead_time_days"] <= 0).sum())},
    {"Check": "Medicine reorder level < 0",
     "Violations": int((medicines["reorder_level"] < 0).sum())},
    {"Check": "Medicine safety stock days < 0",
     "Violations": int((medicines["safety_stock_days"] < 0).sum())},
]
quantity_report = pd.DataFrame(quantity_results)

print("\nQUANTITY INTEGRITY")
print("=" * 65)
display(quantity_report)
print("Checks performed:", len(quantity_report))
print("Total violations:", int(quantity_report["Violations"].sum()))

for row in quantity_results:
    record("3", row["Check"], row["Violations"], 0, group="Quantities")

display(section_table("3"))

CHRONOLOGICAL CONSISTENCY


,Check,Violations
0,Batch expiry after receipt,0
1,Dispensing before batch receipt,0
2,Dispensing after batch expiry,0
3,Goods receipt before delivery date,0
4,Inspection before physical receipt,0
5,Receipt-item expiry differs from batch expiry,0
6,Batch receipt date differs from goods receipt ...,0


Checks performed: 7
Total violations: 0

QUANTITY INTEGRITY


,Check,Violations
0,Batch quantity received <= 0,0
1,Batch quantity available < 0,0
2,Available quantity exceeds received quantity,0
3,Dispensed quantity <= 0,0
4,Supplier delivery lead time <= 0,0
5,Medicine reorder level < 0,0
6,Medicine safety stock days < 0,0


Checks performed: 7
Total violations: 0

SECTION 3 CONFIRMATION: 17/17 checks passed


,Check,Observed,Expected,Status
0,Date fields checked,21,21,PASS
1,Invalid date values,0,0,PASS
2,Dispensing transactions dated after the invent...,0,0,PASS
3,Batch expiry after receipt,0,0,PASS
4,Dispensing before batch receipt,0,0,PASS
5,Dispensing after batch expiry,0,0,PASS
6,Goods receipt before delivery date,0,0,PASS
7,Inspection before physical receipt,0,0,PASS
8,Receipt-item expiry differs from batch expiry,0,0,PASS
9,Batch receipt date differs from goods receipt ...,0,0,PASS


### Findings: dates and quantities
- All **21 date fields parse** with no invalid values. The recorded periods match the intended coverage: dispensing from 1 November 2024 to the 9 November 2026 snapshot (551 of the 42,617 transactions are the snapshot-period dispensing of 1 to 9 November, none is dated later), forecast evaluation from May to October 2026, and November forecasts for 1 to 30 November 2026.
- All **seven chronology rules pass**: batches never expire before receipt, nothing is dispensed before receipt or after expiry, receipts and inspections follow delivery, and receipt-item dates agree with their batches and goods receipts.
- All **seven quantity rules pass**: no zero or negative received or dispensed quantities, no negative stock, no batch holding more than was received, positive lead times, and non-negative reorder levels and safety-stock days.

---
## 4. FEFO compliance
**Purpose:** confirm that dispensing followed First Expiry First Out, using the rules in the methodology. A batch is eligible only if it is received, unexpired, has stock, and belongs to the medicine dispensed. Of the eligible batches, the one that expires first must be used, unless the transaction is a recorded override.

**Method:**
1. *Eligibility audit:* every transaction must use a batch of the same medicine, received on or before the dispensing date and not expired on that date.
2. *Chronological replay:* all transactions are replayed in time order, starting from each batch's received quantity. For every transaction the notebook rebuilds the list of eligible batches, ordered by expiry date, and checks the batch actually used. It also checks stock availability at that moment, which a final batch balance alone cannot show.

**Pass criterion:** zero eligibility violations, zero stock shortages and zero non-override transactions that skipped the earliest-expiring eligible batch. Overrides are expected to use a *later*-expiring batch.

In [10]:
# 4.1  Batch eligibility of every dispensing transaction
transactions = datasets["dispensing_transactions"].copy()
batches = datasets["medicine_batches"].copy()

batch_reference = batches[["batch_id", "medicine_id", "received_date", "expiry_date", "status"]].copy()
batch_reference["received_date"] = pd.to_datetime(batch_reference["received_date"])
batch_reference["expiry_date"] = pd.to_datetime(batch_reference["expiry_date"])
transactions["dispensed_at"] = pd.to_datetime(transactions["dispensed_at"])

fefo_check = transactions.merge(
    batch_reference, on="batch_id", how="left", suffixes=("_transaction", "_batch")
)
dispensing_date = fefo_check["dispensed_at"].dt.normalize()

fefo_results = [
    {"Check": "Transaction medicine differs from batch medicine",
     "Violations": int((fefo_check["medicine_id_transaction"] != fefo_check["medicine_id_batch"]).sum())},
    {"Check": "Batch received after dispensing",
     "Violations": int((fefo_check["received_date"] > dispensing_date).sum())},
    {"Check": "Batch expired before dispensing",
     "Violations": int((fefo_check["expiry_date"] < dispensing_date).sum())},
]
fefo_report = pd.DataFrame(fefo_results)
display(fefo_report)

print("FEFO BATCH ELIGIBILITY SUMMARY")
print("=" * 65)
print("Transactions checked:", len(fefo_check))
print("Total eligibility violations:", int(fefo_report["Violations"].sum()))

print("\nDISPENSING OVERRIDE PROFILE")
print(transactions["is_override"].value_counts(dropna=False).rename_axis("Is Override").to_string())
print("\nBATCH STATUS PROFILE")
print(batches["status"].value_counts(dropna=False).to_string())

for row in fefo_results:
    record("4", row["Check"], row["Violations"], 0,
           group="FEFO eligibility", key="FEFO eligibility violations")

,Check,Violations
0,Transaction medicine differs from batch medicine,0
1,Batch received after dispensing,0
2,Batch expired before dispensing,0


FEFO BATCH ELIGIBILITY SUMMARY
Transactions checked: 42617
Total eligibility violations: 0

DISPENSING OVERRIDE PROFILE
Is Override
False    42414
True       203

BATCH STATUS PROFILE
status
Depleted    2400
Active       100


In [11]:
# 4.2  Chronological FEFO replay (all transactions, in time order)
transactions = datasets["dispensing_transactions"].copy()
batches = datasets["medicine_batches"].copy()

transactions["dispensed_at"] = pd.to_datetime(transactions["dispensed_at"])
batches["received_date"] = pd.to_datetime(batches["received_date"])
batches["expiry_date"] = pd.to_datetime(batches["expiry_date"])

# Every batch starts with its originally received quantity
remaining_stock = dict(zip(batches["batch_id"], batches["quantity_received"]))

# Batches per medicine, in FEFO order (expiry date, then receipt date, then ID)
batch_groups = {
    medicine_id: group.sort_values(["expiry_date", "received_date", "batch_id"]).to_dict("records")
    for medicine_id, group in batches.groupby("medicine_id")
}

transactions = transactions.sort_values(["dispensed_at", "transaction_id"])

replay_results = {
    "transactions_replayed": 0,
    "stock_shortage": 0,
    "no_eligible_batch": 0,
    "selected_batch_not_eligible": 0,
    "non_override_not_earliest_fefo": 0,
    "override_not_earliest_fefo": 0,
    "overrides_without_reason": 0,
}

for tx in transactions.itertuples(index=False):
    replay_results["transactions_replayed"] += 1

    dispensing_date = tx.dispensed_at.normalize()
    medicine_batches = batch_groups.get(tx.medicine_id, [])

    eligible_batches = [
        batch for batch in medicine_batches
        if (batch["received_date"] <= dispensing_date
            and batch["expiry_date"] >= dispensing_date
            and remaining_stock[batch["batch_id"]] > 0)
    ]

    if not eligible_batches:
        replay_results["no_eligible_batch"] += 1
    else:
        earliest_batch_id = eligible_batches[0]["batch_id"]

        if tx.batch_id not in {batch["batch_id"] for batch in eligible_batches}:
            replay_results["selected_batch_not_eligible"] += 1
        elif tx.batch_id != earliest_batch_id:
            if tx.is_override:
                replay_results["override_not_earliest_fefo"] += 1
            else:
                replay_results["non_override_not_earliest_fefo"] += 1

    available_before_dispensing = remaining_stock.get(tx.batch_id, 0)
    if tx.quantity_dispensed > available_before_dispensing:
        replay_results["stock_shortage"] += 1

    if tx.is_override and not str(tx.notes or "").strip():
        replay_results["overrides_without_reason"] += 1

    # Apply the recorded transaction to the selected batch
    remaining_stock[tx.batch_id] = available_before_dispensing - tx.quantity_dispensed

print("CHRONOLOGICAL FEFO REPLAY RESULTS")
print("=" * 65)
for check, count in replay_results.items():
    print(f"{check.replace('_', ' ').title():40} {count:,}")

non_override_count = int((~transactions["is_override"].astype(bool)).sum())
override_count = int(transactions["is_override"].astype(bool).sum())

print("\nFEFO COMPLIANCE SUMMARY")
print("=" * 65)
print("Non-override transactions:", non_override_count)
print("Override transactions:", override_count)

record("4", "Transactions replayed", replay_results["transactions_replayed"],
       len(datasets["dispensing_transactions"]), group="FEFO replay")
record("4", "Stock shortages at the time of dispensing", replay_results["stock_shortage"], 0,
       group="FEFO replay", key="FEFO eligibility violations")
record("4", "Transactions with no eligible batch", replay_results["no_eligible_batch"], 0,
       group="FEFO replay", key="FEFO eligibility violations")
record("4", "Transactions using a batch that was not eligible",
       replay_results["selected_batch_not_eligible"], 0,
       group="FEFO replay", key="FEFO eligibility violations")
record("4", "Non-override transactions that skipped the earliest-expiring eligible batch",
       replay_results["non_override_not_earliest_fefo"], 0,
       group="FEFO replay", key="FEFO eligibility violations")
record("4", "Overrides that used a later-expiring eligible batch (all overrides, by definition)",
       replay_results["override_not_earliest_fefo"], override_count, group="FEFO replay")
record("4", "Overrides without a recorded reason", replay_results["overrides_without_reason"], 0,
       group="FEFO replay")

LIMITS["override_total"] = override_count
display(section_table("4"))

CHRONOLOGICAL FEFO REPLAY RESULTS
Transactions Replayed                    42,617
Stock Shortage                           0
No Eligible Batch                        0
Selected Batch Not Eligible              0
Non Override Not Earliest Fefo           0
Override Not Earliest Fefo               203
Overrides Without Reason                 0

FEFO COMPLIANCE SUMMARY
Non-override transactions: 42414
Override transactions: 203

SECTION 4 CONFIRMATION: 10/10 checks passed


,Check,Observed,Expected,Status
0,Transaction medicine differs from batch medicine,0,0,PASS
1,Batch received after dispensing,0,0,PASS
2,Batch expired before dispensing,0,0,PASS
3,Transactions replayed,42617,42617,PASS
4,Stock shortages at the time of dispensing,0,0,PASS
5,Transactions with no eligible batch,0,0,PASS
6,Transactions using a batch that was not eligible,0,0,PASS
7,Non-override transactions that skipped the ear...,0,0,PASS
8,Overrides that used a later-expiring eligible ...,203,203,PASS
9,Overrides without a recorded reason,0,0,PASS


### Findings: FEFO compliance
- **No eligibility violations.** Every one of the 42,617 transactions used a batch of the correct medicine that had been received and had not expired.
- **The replay confirms FEFO in time order.** No transaction lacked stock or an eligible batch, and all **42,414 non-override transactions** used the earliest-expiring eligible batch.
- **The 203 overrides all used a later-expiring eligible batch and all carry a reason.** The replay counts them as departures from the FEFO order, which is expected because that is what an override is. They are documented exceptions, not violations. Section 5 inspects them.

---
## 5. Dispensing overrides
**Purpose:** an override lets the pharmacist dispense a later-expiring batch instead of the FEFO recommendation. The 203 overrides are the only transactions that depart from FEFO, so they need a closer look: are the reasons recorded, was the chosen batch safe for the whole prescription, and was an earlier-expiring batch actually available?

**Method:**
1. *Reasons:* list the distinct reasons and confirm none is blank.
2. *Prescription safety:* read the prescription duration from the reason text and confirm that the selected batch does not expire before the last day of the prescription (`required_until = dispensing date + duration − 1`).
3. *Necessity:* for each override, look for an earlier-expiring batch that was received, safe for the whole prescription and still in stock, rebuilding stock from the transaction history. Then test whether that earlier stock could have covered the whole quantity.

**Generator rule behind the overrides** (Appendix A): when more than one eligible batch exists, an override happens at random with probability 0.005. It then draws on the second eligible batch first (the other eligible batches cover any remainder, in FEFO order), and it never skips the expiry or prescription-validity checks.

In [12]:
# 5.1  Override reasons and prescription-period safety
overrides = datasets["dispensing_transactions"].loc[
    datasets["dispensing_transactions"]["is_override"] == True
].copy()

overrides["override_reason"] = overrides["notes"].fillna("").astype(str).str.strip()

print("DISPENSING OVERRIDE REASON AUDIT")
print("=" * 70)
print(f"Total override transactions: {len(overrides):,}")
print("Missing or blank reasons:", int(overrides["override_reason"].eq("").sum()))

reason_audit = (
    overrides["override_reason"].value_counts()
    .rename_axis("Override Reason").reset_index(name="Transactions")
)
print(f"Distinct override reasons: {reason_audit.shape[0]}")

with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(reason_audit)

print("REASON LENGTH PROFILE (characters)")
display(overrides["override_reason"].str.len().describe().to_frame("Character Count"))

print("SAMPLE OVERRIDE TRANSACTIONS")
with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(overrides[["transaction_id", "medicine_id", "batch_id",
                       "dispensed_at", "override_reason"]].head(10))

# Prescription-period safety of the selected batch
override_safety = overrides[
    ["transaction_id", "medicine_id", "batch_id", "dispensed_at", "override_reason"]
].copy()
override_safety["dispensed_at"] = pd.to_datetime(override_safety["dispensed_at"])

override_safety["prescription_days"] = (
    override_safety["override_reason"]
    .str.extract(r"Prescription duration:\s*(\d+)\s*days")[0]
    .astype("Int64")
)

override_safety = override_safety.merge(
    datasets["medicine_batches"][["batch_id", "expiry_date"]].copy(),
    on="batch_id", how="left",
)
override_safety["expiry_date"] = pd.to_datetime(override_safety["expiry_date"])

override_safety["required_until"] = (
    override_safety["dispensed_at"].dt.normalize()
    + pd.to_timedelta(override_safety["prescription_days"] - 1, unit="D")
)
override_safety["covers_prescription_period"] = (
    override_safety["expiry_date"] >= override_safety["required_until"]
)

print("\nOVERRIDE PRESCRIPTION-SAFETY AUDIT")
print("=" * 75)
print(f"Override transactions: {len(override_safety):,}")
print("Missing prescription durations:", int(override_safety["prescription_days"].isna().sum()))
print("Selected batches covering the full prescription period:",
      int(override_safety["covers_prescription_period"].sum()))
print("Selected batches failing prescription-period coverage:",
      int((~override_safety["covers_prescription_period"]).sum()))

failed = override_safety.loc[
    ~override_safety["covers_prescription_period"],
    ["transaction_id", "medicine_id", "batch_id", "dispensed_at",
     "prescription_days", "expiry_date", "required_until"],
]
print("\nFAILED RECORDS")
display(failed.head(20))

record("5", "Overrides with a missing or blank reason",
       overrides["override_reason"].eq("").sum(), 0, group="Overrides")
record("5", "Overrides with no readable prescription duration",
       override_safety["prescription_days"].isna().sum(), 0, group="Overrides")
record("5", "Overrides whose selected batch does not cover the prescription period",
       (~override_safety["covers_prescription_period"]).sum(), 0, group="Overrides")

DISPENSING OVERRIDE REASON AUDIT
Total override transactions: 203
Missing or blank reasons: 0
Distinct override reasons: 14


,Override Reason,Transactions
0,Synthetic documented operational override. Prescription duration: 3 days.,20
1,Synthetic documented operational override. Prescription duration: 2 days.,19
2,Synthetic documented operational override. Prescription duration: 9 days.,19
3,Synthetic documented operational override. Prescription duration: 14 days.,17
4,Synthetic documented operational override. Prescription duration: 4 days.,16
5,Synthetic documented operational override. Prescription duration: 1 days.,16
6,Synthetic documented operational override. Prescription duration: 12 days.,15
7,Synthetic documented operational override. Prescription duration: 13 days.,14
8,Synthetic documented operational override. Prescription duration: 5 days.,14
9,Synthetic documented operational override. Prescription duration: 10 days.,12


REASON LENGTH PROFILE (characters)


,Character Count
count,203.000000
mean,73.334975
std,0.473149
min,73.000000
25%,73.000000
50%,73.000000
75%,74.000000
max,74.000000


SAMPLE OVERRIDE TRANSACTIONS


,transaction_id,medicine_id,batch_id,dispensed_at,override_reason
322,TXN0000323,MED001,BAT000112,2025-10-27T09:32:00,Synthetic documented operational override. Prescription duration: 2 days.
524,TXN0000525,MED001,BAT000119,2026-05-27T14:23:00,Synthetic documented operational override. Prescription duration: 4 days.
658,TXN0000659,MED001,BAT000123,2026-10-21T13:07:00,Synthetic documented operational override. Prescription duration: 3 days.
729,TXN0000730,MED002,BAT000126,2025-01-10T11:00:00,Synthetic documented operational override. Prescription duration: 2 days.
1074,TXN0001075,MED002,BAT000136,2026-01-20T12:41:00,Synthetic documented operational override. Prescription duration: 4 days.
1119,TXN0001120,MED002,BAT000140,2026-03-10T12:31:00,Synthetic documented operational override. Prescription duration: 3 days.
1198,TXN0001199,MED002,BAT000136,2026-05-29T13:32:00,Synthetic documented operational override. Prescription duration: 14 days.
1395,TXN0001396,MED003,BAT000151,2025-01-06T12:09:00,Synthetic documented operational override. Prescription duration: 13 days.
1654,TXN0001655,MED003,BAT000161,2026-01-31T09:52:00,Synthetic documented operational override. Prescription duration: 13 days.
1794,TXN0001795,MED003,BAT000167,2026-08-09T13:45:00,Synthetic documented operational override. Prescription duration: 4 days.



OVERRIDE PRESCRIPTION-SAFETY AUDIT
Override transactions: 203
Missing prescription durations: 0
Selected batches covering the full prescription period: 203
Selected batches failing prescription-period coverage: 0

FAILED RECORDS


,transaction_id,medicine_id,batch_id,dispensed_at,prescription_days,expiry_date,required_until


In [13]:
# 5.2  Were the overrides necessary? Earlier-expiring stock at the time of each override
transactions = datasets["dispensing_transactions"].copy()
batches = datasets["medicine_batches"].copy()

transactions["dispensed_at"] = pd.to_datetime(transactions["dispensed_at"])
batches["received_date"] = pd.to_datetime(batches["received_date"])
batches["expiry_date"] = pd.to_datetime(batches["expiry_date"])

transactions = transactions.sort_values(["dispensed_at", "transaction_id"]).reset_index(drop=True)
transactions["event_order"] = range(len(transactions))

batch_quantities = batches.set_index("batch_id")["quantity_received"].to_dict()
quantity_by_transaction = transactions.set_index("transaction_id")["quantity_dispensed"]

override_events = override_safety.merge(
    transactions[["transaction_id", "event_order"]], on="transaction_id", how="left"
)


def earlier_batches(event):
    """Batches expiring earlier than the selected one that were safe for the prescription."""
    return batches[
        (batches["medicine_id"] == event["medicine_id"])
        & (batches["received_date"] <= event["dispensed_at"].normalize())
        & (batches["expiry_date"] >= event["required_until"])
        & (batches["expiry_date"] < event["expiry_date"])
    ].copy()


def stock_before(event, candidates):
    """Stock of each candidate batch just before the override (received minus earlier dispensing)."""
    prior_usage = transactions.loc[
        transactions["event_order"] < event["event_order"]
    ].groupby("batch_id")["quantity_dispensed"].sum()
    return (candidates["batch_id"].map(batch_quantities)
            - candidates["batch_id"].map(prior_usage).fillna(0))


# (a) Was any earlier-expiring batch available and in stock?
availability = []
for _, event in override_events.iterrows():
    candidates = earlier_batches(event)
    if not candidates.empty:
        candidates["stock_before_transaction"] = stock_before(event, candidates)
        candidates = candidates[candidates["stock_before_transaction"] > 0]
    availability.append({
        "transaction_id": event["transaction_id"],
        "earlier_safe_batch_available": not candidates.empty,
        "earlier_safe_batch_count": len(candidates),
        "earliest_alternative_expiry": candidates["expiry_date"].min() if not candidates.empty else pd.NaT,
    })
override_necessity = pd.DataFrame(availability)

print("OVERRIDE NECESSITY AUDIT")
print("=" * 75)
print(f"Overrides examined: {len(override_necessity):,}")
print("Overrides with an earlier safe batch available:",
      int(override_necessity["earlier_safe_batch_available"].sum()))
print("Overrides with no earlier safe batch available:",
      int((~override_necessity["earlier_safe_batch_available"]).sum()))
print("\nSAMPLE OVERRIDES WITH EARLIER SAFE STOCK")
display(override_necessity.loc[override_necessity["earlier_safe_batch_available"]].head(10))

# (b) Could one earlier batch have covered the whole quantity?
quantity_checks = []
for _, event in override_events.iterrows():
    event_quantity = quantity_by_transaction.loc[event["transaction_id"]]
    candidates = earlier_batches(event)
    candidates["stock_before_transaction"] = stock_before(event, candidates)
    candidates = candidates[candidates["stock_before_transaction"] >= event_quantity]
    quantity_checks.append({
        "transaction_id": event["transaction_id"],
        "quantity_dispensed": event_quantity,
        "sufficient_earlier_batch_available": not candidates.empty,
        "sufficient_earlier_batch_count": len(candidates),
    })
override_quantity_audit = pd.DataFrame(quantity_checks)

print("\nOVERRIDE ALTERNATIVE-STOCK QUANTITY AUDIT")
print("=" * 75)
print(f"Overrides examined: {len(override_quantity_audit):,}")
print("Overrides with sufficient earlier FEFO stock:",
      int(override_quantity_audit["sufficient_earlier_batch_available"].sum()))
print("Overrides without sufficient earlier FEFO stock:",
      int((~override_quantity_audit["sufficient_earlier_batch_available"]).sum()))

# (c) The stock-constrained overrides in detail
constrained_ids = override_quantity_audit.loc[
    ~override_quantity_audit["sufficient_earlier_batch_available"], "transaction_id"
].tolist()

details = []
for _, event in override_events[override_events["transaction_id"].isin(constrained_ids)].iterrows():
    earlier = earlier_batches(event)
    earlier["stock_before_transaction"] = stock_before(event, earlier)

    selected = batches[batches["batch_id"] == event["batch_id"]].copy()
    selected["stock_before_transaction"] = stock_before(event, selected)

    details.append({
        "transaction_id": event["transaction_id"],
        "medicine_id": event["medicine_id"],
        "selected_batch_id": event["batch_id"],
        "quantity_dispensed": quantity_by_transaction.loc[event["transaction_id"]],
        "earlier_safe_batch_count": len(earlier),
        "earlier_safe_stock_total": earlier["stock_before_transaction"].clip(lower=0).sum(),
        "largest_earlier_safe_batch_stock": (
            earlier["stock_before_transaction"].max() if not earlier.empty else 0),
        "selected_batch_stock_before": selected["stock_before_transaction"].iloc[0],
    })
stock_constraint_audit = pd.DataFrame(details)

print("\nSTOCK-CONSTRAINED OVERRIDES (earlier stock could not cover the quantity)")
display(stock_constraint_audit)

no_earlier = int((~override_necessity["earlier_safe_batch_available"]).sum())
not_sufficient = int((~override_quantity_audit["sufficient_earlier_batch_available"]).sum())
record("5", "Overrides examined", len(override_necessity), LIMITS["override_total"], group="Overrides")
LIMITS["override_no_earlier_batch"] = no_earlier
LIMITS["override_insufficient_alternative"] = not_sufficient

display(section_table("5"))
print("Observation (not a failure): overrides with an earlier safe batch available:",
      len(override_necessity) - no_earlier,
      "| overrides where no earlier batch could cover the whole quantity:", not_sufficient)

OVERRIDE NECESSITY AUDIT
Overrides examined: 203
Overrides with an earlier safe batch available: 203
Overrides with no earlier safe batch available: 0

SAMPLE OVERRIDES WITH EARLIER SAFE STOCK


,transaction_id,earlier_safe_batch_available,earlier_safe_batch_count,earliest_alternative_expiry
0,TXN0000323,True,1,2027-07-26
1,TXN0000525,True,1,2028-06-19
2,TXN0000659,True,1,2028-12-27
3,TXN0000730,True,1,2026-07-18
4,TXN0001075,True,1,2028-01-06
5,TXN0001120,True,1,2028-01-19
6,TXN0001199,True,1,2027-12-29
7,TXN0001396,True,1,2026-09-20
8,TXN0001655,True,1,2027-08-29
9,TXN0001795,True,1,2028-08-10



OVERRIDE ALTERNATIVE-STOCK QUANTITY AUDIT
Overrides examined: 203
Overrides with sufficient earlier FEFO stock: 199
Overrides without sufficient earlier FEFO stock: 4

STOCK-CONSTRAINED OVERRIDES (earlier stock could not cover the quantity)


,transaction_id,medicine_id,selected_batch_id,quantity_dispensed,earlier_safe_batch_count,earlier_safe_stock_total,largest_earlier_safe_batch_stock,selected_batch_stock_before
0,TXN0003315,MED007,BAT000256,8,12,7,7,54.0
1,TXN0005208,MED012,BAT000376,6,12,4,4,99.0
2,TXN0030655,MED072,BAT001812,5,8,4,4,120.0
3,TXN0032314,MED077,BAT001935,5,10,1,1,120.0



SECTION 5 CONFIRMATION: 4/4 checks passed


,Check,Observed,Expected,Status
0,Overrides with a missing or blank reason,0,0,PASS
1,Overrides with no readable prescription duration,0,0,PASS
2,Overrides whose selected batch does not cover ...,0,0,PASS
3,Overrides examined,203,203,PASS


Observation (not a failure): overrides with an earlier safe batch available: 203 | overrides where no earlier batch could cover the whole quantity: 4


### Findings: overrides
- **All 203 overrides have a recorded reason.** There are 14 distinct reasons, all of the form *"Synthetic documented operational override. Prescription duration: N days."* with N from 1 to 14.
- **Every override was prescription-safe.** The selected batch covers the whole prescription period for all 203 (0 failures).
- **An earlier-expiring safe batch existed for all 203 overrides.** For 199 of them, one earlier batch could have supplied the full quantity as well. For **4 overrides** (`TXN0003315`, `TXN0005208`, `TXN0030655` and `TXN0032314`) it could not: the earlier stock in total (7, 4, 4 and 1 units) was less than the quantity dispensed (8, 6, 5 and 5 units).
- **Interpretation:** the generator applies overrides at random (probability 0.5%), not because of a stock shortage, so these are synthetic test scenarios and not real clinical events. They exercise the application's override-with-reason rule. They are accepted as documented exceptions. The four stock-constrained cases are recorded in Section 10 as an observation.

---
## 6. Supplier master data
**Purpose:** confirm that the 15 suppliers are complete, unique and correctly formatted, and show what the data does and does not vary.

**Method:** the file is read again with `phone_number` as **text**, because the default load turns it into an integer (Section 1). The cell then checks required fields, uniqueness, the email format, the Kenyan mobile format (`+2547` followed by 8 digits), active status, delivery lead times and address diversity.

**Pass criterion:** no missing or duplicate values, and all emails and phone numbers valid. Uniform lead times and addresses are not failures. They are limitations to document.

In [14]:
# 6.1  Supplier master-data audit (phone_number read as text)
supplier_audit = pd.read_csv(
    DATASET_DIR / "suppliers.csv", dtype={"phone_number": "string"}
)

print("SUPPLIER MASTER-DATA AUDIT")
print("=" * 80)

required_columns = ["supplier_id", "supplier_name", "contact_person", "email",
                    "phone_number", "address", "delivery_lead_time_days", "is_active"]

print("\n1. REQUIRED FIELDS")
missing_total = 0
for column in required_columns:
    missing = int(supplier_audit[column].isna().sum())
    blank = int(supplier_audit[column].astype("string").str.strip().eq("").sum())
    missing_total += missing + blank
    print(f"{column}: missing={missing}, blank={blank}")

print("\n2. UNIQUENESS")
duplicate_total = 0
for column in ["supplier_id", "supplier_name", "email", "phone_number"]:
    duplicates = int(supplier_audit[column].duplicated().sum())
    duplicate_total += duplicates
    print(f"{column}: duplicates={duplicates}")

print("\n3. CONTACT FORMAT")
email_valid = supplier_audit["email"].str.fullmatch(
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", na=False
)
phone_valid = supplier_audit["phone_number"].str.fullmatch(r"\+2547\d{8}", na=False)
print(f"Valid email formats: {email_valid.sum()}/{len(supplier_audit)}")
print(f"Valid Kenyan mobile formats: {phone_valid.sum()}/{len(supplier_audit)}")

print("\n4. ACTIVE STATUS")
print(supplier_audit["is_active"].value_counts(dropna=False).to_string())

print("\n5. DELIVERY LEAD TIME")
print(supplier_audit["delivery_lead_time_days"].describe().to_string())
distinct_lead_times = sorted(supplier_audit["delivery_lead_time_days"].dropna().unique())
print("Distinct lead times:", [int(value) for value in distinct_lead_times])

print("\n6. ADDRESS DIVERSITY")
print("Distinct addresses:", supplier_audit["address"].nunique())
print(supplier_audit["address"].value_counts().to_string())

print("\n7. PHONE NUMBER SAMPLE AS EXPORTED")
print(supplier_audit[["supplier_id", "phone_number"]].head(10).to_string(index=False))

record("6", "Missing or blank required supplier fields", missing_total, 0, group="Suppliers")
record("6", "Duplicate supplier IDs, names, emails or phone numbers", duplicate_total, 0, group="Suppliers")
record("6", "Valid email formats", email_valid.sum(), len(supplier_audit), group="Suppliers")
record("6", "Valid Kenyan mobile phone formats", phone_valid.sum(), len(supplier_audit), group="Suppliers")

LIMITS["lead_time_days"] = int(distinct_lead_times[0]) if len(distinct_lead_times) == 1 else None
LIMITS["distinct_addresses"] = int(supplier_audit["address"].nunique())
display(section_table("6"))

SUPPLIER MASTER-DATA AUDIT

1. REQUIRED FIELDS
supplier_id: missing=0, blank=0
supplier_name: missing=0, blank=0
contact_person: missing=0, blank=0
email: missing=0, blank=0
phone_number: missing=0, blank=0
address: missing=0, blank=0
delivery_lead_time_days: missing=0, blank=0
is_active: missing=0, blank=0

2. UNIQUENESS
supplier_id: duplicates=0
supplier_name: duplicates=0
email: duplicates=0
phone_number: duplicates=0

3. CONTACT FORMAT
Valid email formats: 15/15
Valid Kenyan mobile formats: 15/15

4. ACTIVE STATUS
is_active
True    15

5. DELIVERY LEAD TIME
count    15.0
mean      7.0
std       0.0
min       7.0
25%       7.0
50%       7.0
75%       7.0
max       7.0
Distinct lead times: [7]

6. ADDRESS DIVERSITY
Distinct addresses: 1
address
Fictional Business Centre, Nairobi, Kenya    15

7. PHONE NUMBER SAMPLE AS EXPORTED
supplier_id  phone_number
     SUP001 +254714228155
     SUP002 +254769624594
     SUP003 +254780214626
     SUP004 +254730717331
     SUP005 +254711939083
   

,Check,Observed,Expected,Status
0,Missing or blank required supplier fields,0,0,PASS
1,"Duplicate supplier IDs, names, emails or phone...",0,0,PASS
2,Valid email formats,15,15,PASS
3,Valid Kenyan mobile phone formats,15,15,PASS


### Findings: supplier master data
- All 15 suppliers are **complete, unique, active and correctly formatted**: 15 of 15 valid emails and 15 of 15 valid Kenyan mobile numbers. The phone numbers are 13-character text values such as `+254714228155`, which fit the application's `phone_number` field (maximum 15 characters).
- **Limitation:** every supplier has the **same 7-day lead time** and the **same fictional address**, so the data cannot show how different supplier lead times change procurement. This follows from the generator's constants, and it is a documented limitation, not an error.

---
## 7. Procurement and receiving reconciliation
**Purpose:** follow the supply chain from purchase order to goods receipt, receipt item, batch and final stock balance, and confirm that every link agrees.

**Method (four reconciliations):**
1. *Purchase order to goods receipt:* every receipt refers to a purchase order that was placed with the supplier. Orders without a receipt must each have an explanation (their status, or a delivery that is still expected after the 9 November snapshot).
2. *Goods receipt to batch:* every receipt item is linked to a batch with the same medicine, batch number, quantity, expiry date and storage location, and each receipt's total equals the quantity ordered.
3. *Opening inventory:* batches with no receipt item must be identified as opening inventory (initial batch numbers, received before the first delivery, already depleted).
4. *Batch stock ledger:* for every batch, available quantity = received − all dispensed, and the batch status matches (Depleted at zero, otherwise Active).

**Pass criterion:** zero discrepancies, and every unreceived order and every batch without a receipt explained.

In [15]:
# 7.1  Purchase order → goods receipt reconciliation
goods_receipts = datasets["goods_receipts"]
purchase_orders = datasets["purchase_orders"]
suppliers = datasets["suppliers"]

po_receipt = goods_receipts.merge(
    purchase_orders[["purchase_order_id", "medicine_id", "supplier_id",
                     "quantity_ordered", "order_date", "status"]],
    on="purchase_order_id", how="left", indicator=True, validate="one_to_one",
)
po_receipt["order_date"] = pd.to_datetime(po_receipt["order_date"], errors="coerce")
po_receipt["delivery_date"] = pd.to_datetime(po_receipt["delivery_date"], errors="coerce")

checks = {
    "Receipts referencing existing purchase orders": (po_receipt["_merge"] == "both").sum(),
    "Receipts linked to orders not placed with supplier": (po_receipt["status"] != "Placed with Supplier").sum(),
    "Delivery dates before order dates": (po_receipt["delivery_date"] < po_receipt["order_date"]).sum(),
    "Missing order dates": po_receipt["order_date"].isna().sum(),
    "Missing delivery dates": po_receipt["delivery_date"].isna().sum(),
    "Duplicate purchase orders in receipt records": goods_receipts["purchase_order_id"].duplicated().sum(),
}

print("PURCHASE ORDER–GOODS RECEIPT RECONCILIATION")
print("=" * 80)
print("Goods receipts:", len(goods_receipts))
print("Purchase orders:", len(purchase_orders))
print(f"Purchase orders without receipts: {len(purchase_orders) - len(goods_receipts):,}")
print("\nPURCHASE ORDER STATUS")
print(purchase_orders["status"].value_counts(dropna=False).to_string())
print("\nRECONCILIATION CHECKS")
for check, result in checks.items():
    print(f"{check}: {result:,}")
print("\nSAMPLE MATCHED RECORDS")
print(po_receipt[["goods_receipt_id", "purchase_order_id", "medicine_id", "supplier_id",
                  "quantity_ordered", "order_date", "delivery_date", "status"]].head(10).to_string(index=False))

# Purchase orders that have no goods receipt
unreceived_orders = purchase_orders.merge(
    goods_receipts[["purchase_order_id"]],
    on="purchase_order_id", how="left", indicator=True, validate="one_to_one",
)
unreceived_orders = unreceived_orders[unreceived_orders["_merge"] == "left_only"].drop(columns="_merge")

print("\nPURCHASE ORDERS WITHOUT GOODS RECEIPTS")
print("=" * 80)
print("Total:", len(unreceived_orders))
print("\nSTATUS BREAKDOWN")
print(unreceived_orders["status"].value_counts(dropna=False).to_string())

placed_without_receipt = unreceived_orders[unreceived_orders["status"] == "Placed with Supplier"]
print("\nUNRECEIVED ORDERS MARKED AS PLACED WITH SUPPLIER")
print("Count:", len(placed_without_receipt))

# Expected delivery of the outstanding orders (order date + supplier lead time)
outstanding_orders = placed_without_receipt.merge(
    suppliers[["supplier_id", "delivery_lead_time_days"]],
    on="supplier_id", how="left", validate="many_to_one",
)
outstanding_orders["order_date"] = pd.to_datetime(outstanding_orders["order_date"])
outstanding_orders["expected_delivery_date"] = (
    outstanding_orders["order_date"].dt.normalize()
    + pd.to_timedelta(outstanding_orders["delivery_lead_time_days"], unit="D")
)
outstanding_orders["days_until_expected_delivery"] = (
    outstanding_orders["expected_delivery_date"] - SNAPSHOT_DATE
).dt.days

print("\nOUTSTANDING DELIVERY VERIFICATION")
print(outstanding_orders[["purchase_order_id", "supplier_id", "quantity_ordered", "order_date",
                          "delivery_lead_time_days", "expected_delivery_date",
                          "days_until_expected_delivery"]].to_string(index=False))

after_snapshot = int((outstanding_orders["expected_delivery_date"] > SNAPSHOT_DATE).sum())
print("\nSUMMARY")
print("Orders:", len(outstanding_orders))
print("Expected delivery after snapshot:", after_snapshot)
print("Expected delivery on snapshot date:",
      int((outstanding_orders["expected_delivery_date"] == SNAPSHOT_DATE).sum()))
print("Expected delivery before snapshot:",
      int((outstanding_orders["expected_delivery_date"] < SNAPSHOT_DATE).sum()))

explained_by_status = int(
    unreceived_orders["status"].isin(["Pending Admin Review", "Approved", "Rejected"]).sum()
)
group = "PO–goods receipt reconciliation"
record("7", "Receipts referencing existing purchase orders",
       checks["Receipts referencing existing purchase orders"], len(goods_receipts), group=group)
for name in ["Receipts linked to orders not placed with supplier", "Delivery dates before order dates",
             "Missing order dates", "Missing delivery dates",
             "Duplicate purchase orders in receipt records"]:
    record("7", name, checks[name], 0, group=group)
record("7", "Unreceived orders not explained by their status or by a delivery expected after the snapshot",
       len(unreceived_orders) - explained_by_status - after_snapshot, 0, group=group)

LIMITS["po_without_receipt"] = len(unreceived_orders)
LIMITS["po_not_placed"] = explained_by_status
LIMITS["po_outstanding"] = after_snapshot

PURCHASE ORDER–GOODS RECEIPT RECONCILIATION
Goods receipts: 2400
Purchase orders: 2455
Purchase orders without receipts: 55

PURCHASE ORDER STATUS
status
Placed with Supplier    2410
Pending Admin Review      26
Approved                  12
Rejected                   7

RECONCILIATION CHECKS
Receipts referencing existing purchase orders: 2,400
Receipts linked to orders not placed with supplier: 0
Delivery dates before order dates: 0
Missing order dates: 0
Missing delivery dates: 0
Duplicate purchase orders in receipt records: 0

SAMPLE MATCHED RECORDS
goods_receipt_id purchase_order_id medicine_id supplier_id  quantity_ordered          order_date delivery_date               status
        GR000001          PO000001      MED001      SUP005                55 2024-10-25 09:00:00    2024-11-01 Placed with Supplier
        GR000002          PO000002      MED001      SUP008               127 2024-11-24 09:00:00    2024-12-01 Placed with Supplier
        GR000003          PO000003      MED001

In [16]:
# 7.2  Goods receipt → medicine batch reconciliation
goods_receipt_items = datasets["goods_receipt_items"]
medicine_batches = datasets["medicine_batches"]

receipt_batch = goods_receipt_items.merge(
    goods_receipts[["goods_receipt_id", "purchase_order_id"]],
    on="goods_receipt_id", how="left", validate="many_to_one",
)
receipt_batch = receipt_batch.merge(
    purchase_orders[["purchase_order_id", "medicine_id", "quantity_ordered"]],
    on="purchase_order_id", how="left", validate="many_to_one",
)
receipt_batch = receipt_batch.merge(
    medicine_batches[["batch_id", "medicine_id", "batch_number", "quantity_received",
                      "expiry_date", "storage_location"]],
    left_on="medicine_batch_id", right_on="batch_id", how="left",
    suffixes=("_receipt", "_batch"), indicator="batch_match", validate="many_to_one",
)

receipt_quantity_totals = (
    goods_receipt_items.groupby("goods_receipt_id", as_index=False)["quantity_received"]
    .sum().rename(columns={"quantity_received": "total_received"})
)
receipt_order = goods_receipts.merge(
    purchase_orders[["purchase_order_id", "quantity_ordered"]],
    on="purchase_order_id", how="left", validate="one_to_one",
).merge(receipt_quantity_totals, on="goods_receipt_id", how="left", validate="one_to_one")

checks = {
    "Receipt items linked to existing batches": receipt_batch["batch_match"].eq("both").sum(),
    "Receipt item medicine differs from purchase order medicine":
        receipt_batch["medicine_id_receipt"].ne(receipt_batch["medicine_id_batch"]).sum(),
    "Batch number differs from receipt item":
        receipt_batch["batch_number_receipt"].ne(receipt_batch["batch_number_batch"]).sum(),
    "Received quantity differs from batch quantity":
        receipt_batch["quantity_received_receipt"].ne(receipt_batch["quantity_received_batch"]).sum(),
    "Expiry date differs from batch":
        pd.to_datetime(receipt_batch["expiry_date_receipt"]).ne(
            pd.to_datetime(receipt_batch["expiry_date_batch"])).sum(),
    "Storage location differs from batch":
        receipt_batch["storage_location_receipt"].ne(receipt_batch["storage_location_batch"]).sum(),
    "Receipt totals differ from purchase order quantities":
        receipt_order["total_received"].ne(receipt_order["quantity_ordered"]).sum(),
}

print("GOODS RECEIPT–BATCH RECONCILIATION")
print("=" * 80)
print(f"Goods receipts: {len(goods_receipts):,}")
print(f"Goods receipt items: {len(goods_receipt_items):,}")
print(f"Medicine batches: {len(medicine_batches):,}")
print("\nRECONCILIATION CHECKS")
for check, result in checks.items():
    print(f"{check}: {result:,}")
print("\nRECEIPT ITEM COUNT PER GOODS RECEIPT")
print(goods_receipt_items.groupby("goods_receipt_id").size().value_counts().sort_index().to_string())
print("\nSAMPLE RECONCILED RECORDS")
print(receipt_order[["goods_receipt_id", "purchase_order_id", "quantity_ordered",
                     "total_received"]].head(10).to_string(index=False))

group = "Goods receipt–batch reconciliation"
record("7", "Receipt items linked to existing batches",
       checks["Receipt items linked to existing batches"], len(goods_receipt_items), group=group)
for name, result in checks.items():
    if name == "Receipt items linked to existing batches":
        continue
    quantity_check = name in ("Received quantity differs from batch quantity",
                              "Receipt totals differ from purchase order quantities")
    record("7", name, result, 0, group=group,
           key="Goods receipt–batch quantity discrepancies" if quantity_check else None)

GOODS RECEIPT–BATCH RECONCILIATION
Goods receipts: 2,400
Goods receipt items: 2,400
Medicine batches: 2,500

RECONCILIATION CHECKS
Receipt items linked to existing batches: 2,400
Receipt item medicine differs from purchase order medicine: 0
Batch number differs from receipt item: 0
Received quantity differs from batch quantity: 0
Expiry date differs from batch: 0
Storage location differs from batch: 0
Receipt totals differ from purchase order quantities: 0

RECEIPT ITEM COUNT PER GOODS RECEIPT
1    2400

SAMPLE RECONCILED RECORDS
goods_receipt_id purchase_order_id  quantity_ordered  total_received
        GR000001          PO000001                55              55
        GR000002          PO000002               127             127
        GR000003          PO000003               143             143
        GR000004          PO000004               102             102
        GR000005          PO000005               149             149
        GR000006          PO000006               1

In [17]:
# 7.3  Opening inventory and batch stock ledger
medicine_batches = datasets["medicine_batches"]
goods_receipt_items = datasets["goods_receipt_items"]
dispensing_transactions = datasets["dispensing_transactions"]

# (a) Batches that have no goods receipt item
batch_receipt_audit = medicine_batches.merge(
    goods_receipt_items[["medicine_batch_id"]],
    left_on="batch_id", right_on="medicine_batch_id",
    how="left", indicator=True, validate="one_to_one",
)
unmatched_batches = batch_receipt_audit[batch_receipt_audit["_merge"] == "left_only"].copy()
matched_batches = batch_receipt_audit[batch_receipt_audit["_merge"] == "both"]

print("MEDICINE BATCH–GOODS RECEIPT COVERAGE")
print("=" * 80)
print(f"Total medicine batches: {len(medicine_batches):,}")
print(f"Batches linked to receipt items: {len(matched_batches):,}")
print(f"Batches without receipt items: {len(unmatched_batches):,}")
print("\nUNMATCHED BATCHES BY STATUS")
print(unmatched_batches["status"].value_counts(dropna=False).to_string())
print("\nUNMATCHED BATCH QUANTITIES")
print(unmatched_batches[["quantity_received", "quantity_available"]].describe().to_string())

unmatched_batches["received_date"] = pd.to_datetime(unmatched_batches["received_date"], errors="coerce")
print("\nUNMATCHED BATCH DATE RANGE")
print("Earliest:", unmatched_batches["received_date"].min())
print("Latest:", unmatched_batches["received_date"].max())
print("\nUNMATCHED BATCH SAMPLE")
print(unmatched_batches[["batch_id", "medicine_id", "batch_number", "quantity_received",
                         "quantity_available", "received_date", "expiry_date",
                         "status"]].head(10).to_string(index=False))

# Opening inventory = initial batch number, received before the first delivery, already depleted
first_delivery = pd.to_datetime(goods_receipts["delivery_date"]).min()
is_opening_inventory = (
    unmatched_batches["batch_number"].str.contains("-INITIAL-", regex=False)
    & (unmatched_batches["received_date"] < first_delivery)
    & (unmatched_batches["quantity_available"] == 0)
    & (unmatched_batches["status"] == "Depleted")
)

group = "Opening inventory reconciliation"
record("7", "Batches without a receipt item that are not depleted opening inventory",
       (~is_opening_inventory).sum(), 0, group=group)
record("7", "Batches explained by a receipt item or by opening inventory",
       len(matched_batches) + int(is_opening_inventory.sum()), len(medicine_batches), group=group)
LIMITS["opening_inventory_batches"] = int(is_opening_inventory.sum())

# (b) Batch stock ledger: received − dispensed must equal available
dispensed_by_batch = (
    dispensing_transactions.groupby("batch_id", as_index=False)["quantity_dispensed"]
    .sum().rename(columns={"quantity_dispensed": "total_dispensed"})
)
stock_reconciliation = medicine_batches.merge(
    dispensed_by_batch, on="batch_id", how="left", validate="one_to_one"
)
stock_reconciliation["total_dispensed"] = stock_reconciliation["total_dispensed"].fillna(0)
stock_reconciliation["expected_available"] = (
    stock_reconciliation["quantity_received"] - stock_reconciliation["total_dispensed"]
)
stock_reconciliation["stock_difference"] = (
    stock_reconciliation["quantity_available"] - stock_reconciliation["expected_available"]
)
stock_reconciliation["expected_status"] = stock_reconciliation["expected_available"].map(
    lambda quantity: "Depleted" if quantity == 0 else "Active"
)

stock_discrepancies = int(stock_reconciliation["stock_difference"].ne(0).sum())
negative_expected = int(stock_reconciliation["expected_available"].lt(0).sum())
status_mismatch = int(stock_reconciliation["status"].ne(stock_reconciliation["expected_status"]).sum())

print("\nBATCH INVENTORY STOCK RECONCILIATION")
print("=" * 80)
print("Total batches:", len(stock_reconciliation))
print("Batches with dispensing history:", int(stock_reconciliation["total_dispensed"].gt(0).sum()))
print("Batches without dispensing history:", int(stock_reconciliation["total_dispensed"].eq(0).sum()))
print("\nSTOCK RECONCILIATION CHECKS")
print("Batches with matching available stock:", int(stock_reconciliation["stock_difference"].eq(0).sum()))
print("Batches with stock discrepancies:", stock_discrepancies)
print("Batches with negative expected stock:", negative_expected)
print("Batches with status inconsistent with stock:", status_mismatch)
print("\nSTOCK DIFFERENCE PROFILE")
print(stock_reconciliation["stock_difference"].value_counts().head(10).to_string())
print("\nDISCREPANCY SAMPLE")
print(stock_reconciliation.loc[
    stock_reconciliation["stock_difference"].ne(0),
    ["batch_id", "quantity_received", "total_dispensed", "quantity_available",
     "expected_available", "stock_difference", "status"],
].head(15).to_string(index=False))

group = "Batch stock ledger reconciliation"
record("7", "Batches whose available stock differs from received minus dispensed",
       stock_discrepancies, 0, group=group, key="Batch stock discrepancies")
record("7", "Batches with negative expected stock", negative_expected, 0, group=group)
record("7", "Batches whose status is inconsistent with their stock", status_mismatch, 0, group=group)

display(section_table("7"))

MEDICINE BATCH–GOODS RECEIPT COVERAGE
Total medicine batches: 2,500
Batches linked to receipt items: 2,400
Batches without receipt items: 100

UNMATCHED BATCHES BY STATUS
status
Depleted    100

UNMATCHED BATCH QUANTITIES
       quantity_received  quantity_available
count         100.000000               100.0
mean          128.850000                 0.0
std            61.626502                 0.0
min            23.000000                 0.0
25%            77.500000                 0.0
50%           132.000000                 0.0
75%           185.000000                 0.0
max           223.000000                 0.0

UNMATCHED BATCH DATE RANGE
Earliest: 2024-10-15 00:00:00
Latest: 2024-10-15 00:00:00

UNMATCHED BATCH SAMPLE
 batch_id medicine_id       batch_number  quantity_received  quantity_available received_date expiry_date   status
BAT000001      MED001 MED001-INITIAL-001                207                   0    2024-10-15  2026-07-15 Depleted
BAT000002      MED002 MED002-INIT

,Check,Observed,Expected,Status
0,Receipts referencing existing purchase orders,2400,2400,PASS
1,Receipts linked to orders not placed with supp...,0,0,PASS
2,Delivery dates before order dates,0,0,PASS
3,Missing order dates,0,0,PASS
4,Missing delivery dates,0,0,PASS
5,Duplicate purchase orders in receipt records,0,0,PASS
6,Unreceived orders not explained by their statu...,0,0,PASS
7,Receipt items linked to existing batches,2400,2400,PASS
8,Receipt item medicine differs from purchase or...,0,0,PASS
9,Batch number differs from receipt item,0,0,PASS


### Findings: procurement and receiving
- **Purchase orders and receipts agree.** All 2,400 goods receipts refer to an existing purchase order that was placed with the supplier, with no delivery before its order date and no duplicate receipts.
- **The 55 purchase orders without a receipt are fully explained:** 26 *Pending Admin Review*, 12 *Approved* and 7 *Rejected* have not reached the delivery stage, and 10 *Placed with Supplier* orders (made on 9 November 2026 with a 7-day lead time) are expected on **16 November 2026**, after the snapshot. They are legitimately outstanding and must not be counted as received stock in the November inventory calculations.
- **Receipts and batches agree.** Each of the 2,400 receipt items is linked to one batch with the same medicine, batch number, quantity, expiry date and storage location. Every receipt has exactly one item, and every receipt total equals the quantity ordered.
- **The 100 batches without receipts are opening inventory:** their batch numbers contain `INITIAL`, they were received on 15 October 2024 (before the first delivery on 1 November 2024) and all are depleted.
- **The stock ledger balances.** For all 2,500 batches, available quantity equals received minus dispensed, and the status is consistent with the stock. No batch ever has negative stock.

---
## 8. Forecasts, recommendations and purchase orders
**Purpose:** confirm that the November procurement recommendations are built correctly from the November forecasts and the FEFO-eligible stock, and that every purchase order traces back to a recommendation.

**Method (four reconciliations):**
1. *Eligible stock:* recalculate the FEFO-eligible stock of each medicine from the batch file and compare it with `eligible_stock_quantity` in the recommendation. A batch counts only if it is **Active, has stock, and stays valid through the 14-day prescription period** (expiry on or after the snapshot + 13 days, as in the generator's `batch_is_eligible`).
2. *Forecast link:* each November recommendation must reference the forecast of the same medicine and carry the same forecast demand.
3. *Formula:* recalculate the three procurement figures independently and compare:
   - projected stock = eligible stock + incoming stock
   - target stock = forecast demand ÷ 30 × (lead time + 30 + safety-stock days), rounded half-up
   - recommended quantity = max(target − projected, 0)
4. *Purchase orders:* each purchase order links one-to-one to a recommendation with the same medicine and quantity.

**Pass criterion:** zero discrepancies in every reconciliation.

In [18]:
# 8.1  FEFO-eligible stock and forecast link of the November recommendations
procurement_recommendations = datasets["procurement_recommendations"]
november_recommendations = procurement_recommendations[
    procurement_recommendations["forecast_id"].notna()
].copy()
november_forecasts = datasets["demand_forecasts_november"]

# Recalculate FEFO-eligible stock per medicine from the batch file
batches = datasets["medicine_batches"].copy()
batches["expiry_date"] = pd.to_datetime(batches["expiry_date"])
required_valid_until = SNAPSHOT_DATE + pd.Timedelta(days=MAX_PRESCRIPTION_DAYS - 1)

eligible_now = batches[
    (batches["status"] == "Active")
    & (batches["quantity_available"] > 0)
    & (batches["expiry_date"] >= required_valid_until)
]
eligible_stock_by_medicine = (
    eligible_now.groupby("medicine_id", as_index=False)["quantity_available"].sum()
    .rename(columns={"quantity_available": "calculated_eligible_stock"})
)

print("November recommendations:", len(november_recommendations))
print("Unique medicines:", november_recommendations["medicine_id"].nunique())
print("Stock must stay valid through:", required_valid_until.date())

stock_comparison = november_recommendations.merge(
    eligible_stock_by_medicine, on="medicine_id", how="left", validate="one_to_one"
)
stock_comparison["calculated_eligible_stock"] = stock_comparison["calculated_eligible_stock"].fillna(0)
stock_comparison["stock_difference"] = (
    stock_comparison["eligible_stock_quantity"] - stock_comparison["calculated_eligible_stock"]
)

print("\nFEFO STOCK–NOVEMBER PROCUREMENT RECONCILIATION")
print("=" * 80)
print("Recommendations checked:", len(stock_comparison))
print("Medicines with matching eligible stock:", int(stock_comparison["stock_difference"].eq(0).sum()))
print("Medicines with stock discrepancies:", int(stock_comparison["stock_difference"].ne(0).sum()))
print("\nDISCREPANCY SAMPLE")
print(stock_comparison.loc[
    stock_comparison["stock_difference"].ne(0),
    ["medicine_id", "eligible_stock_quantity", "calculated_eligible_stock", "stock_difference"],
].head(15).to_string(index=False))

group = "FEFO stock–procurement reconciliation"
record("8", "Recommendations checked", len(stock_comparison), len(november_recommendations), group=group)
record("8", "Medicines whose recommendation stock differs from the recalculated FEFO-eligible stock",
       stock_comparison["stock_difference"].ne(0).sum(), 0, group=group,
       key="November FEFO stock discrepancies")

# Forecast link: same forecast, same medicine, same forecast demand
forecast_reconciliation = november_recommendations.merge(
    november_forecasts[["forecast_id", "medicine_id", "forecast_period_start",
                        "forecast_period_end", "forecasted_demand_quantity"]],
    on=["forecast_id", "medicine_id"], how="left", validate="one_to_one",
    indicator=True, suffixes=("_recommendation", "_forecast"),
)
forecast_reconciliation["demand_difference"] = (
    forecast_reconciliation["forecast_demand_quantity"]
    - forecast_reconciliation["forecasted_demand_quantity"]
)

print("\nFORECAST–PROCUREMENT SYNCHRONIZATION")
print("=" * 80)
print("Recommendations checked:", len(forecast_reconciliation))
print("Matched forecast references:", int(forecast_reconciliation["_merge"].eq("both").sum()))
print("Unmatched forecast references:", int(forecast_reconciliation["_merge"].ne("both").sum()))
print("Matching forecast demand quantities:", int(forecast_reconciliation["demand_difference"].eq(0).sum()))
print("Demand quantity discrepancies:", int(forecast_reconciliation["demand_difference"].ne(0).sum()))
print("\nUNMATCHED OR INCONSISTENT RECORDS")
print(forecast_reconciliation.loc[
    (forecast_reconciliation["_merge"] != "both") | (forecast_reconciliation["demand_difference"] != 0),
    ["recommendation_id", "medicine_id", "forecast_id", "forecast_demand_quantity",
     "forecasted_demand_quantity", "_merge"],
].head(15).to_string(index=False))

group = "Forecast–recommendation synchronization"
record("8", "Recommendations whose forecast reference is not matched",
       forecast_reconciliation["_merge"].ne("both").sum(), 0, group=group)
record("8", "Recommendations whose demand differs from the forecast demand",
       forecast_reconciliation["demand_difference"].ne(0).sum(), 0, group=group,
       key="Forecast demand discrepancies")

November recommendations: 100
Unique medicines: 100
Stock must stay valid through: 2026-11-22

FEFO STOCK–NOVEMBER PROCUREMENT RECONCILIATION
Recommendations checked: 100
Medicines with matching eligible stock: 100
Medicines with stock discrepancies: 0

DISCREPANCY SAMPLE
Empty DataFrame
Columns: [medicine_id, eligible_stock_quantity, calculated_eligible_stock, stock_difference]
Index: []

FORECAST–PROCUREMENT SYNCHRONIZATION
Recommendations checked: 100
Matched forecast references: 100
Unmatched forecast references: 0
Matching forecast demand quantities: 100
Demand quantity discrepancies: 0

UNMATCHED OR INCONSISTENT RECORDS
Empty DataFrame
Columns: [recommendation_id, medicine_id, forecast_id, forecast_demand_quantity, forecasted_demand_quantity, _merge]
Index: []


In [19]:
# 8.2  Procurement formula: independent recalculation
medicines = datasets["medicines"]

procurement_math = november_recommendations.merge(
    medicines[["medicine_id", "safety_stock_days"]],
    on="medicine_id", how="left", validate="one_to_one",
)


def round_half_up(value):
    return int(Decimal(str(value)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


procurement_math["calculated_projected_stock"] = (
    procurement_math["eligible_stock_quantity"] + procurement_math["incoming_stock_quantity"]
)
procurement_math["calculated_target_stock"] = procurement_math.apply(
    lambda row: round_half_up(
        (Decimal(str(row["forecast_demand_quantity"])) / Decimal("30"))
        * (Decimal(str(row["lead_time_days"])) + Decimal("30") + Decimal(str(row["safety_stock_days"])))
    ),
    axis=1,
)
procurement_math["calculated_recommended_quantity"] = (
    procurement_math["calculated_target_stock"] - procurement_math["calculated_projected_stock"]
).clip(lower=0)

print("NOVEMBER PROCUREMENT FORMULA RECONCILIATION")
print("=" * 80)

checks = {
    "Projected stock matches":
        procurement_math["projected_stock_position"] == procurement_math["calculated_projected_stock"],
    "Target stock matches":
        procurement_math["target_stock_quantity"] == procurement_math["calculated_target_stock"],
    "Recommended quantity matches":
        procurement_math["recommended_quantity"] == procurement_math["calculated_recommended_quantity"],
    "No negative recommended quantities": procurement_math["recommended_quantity"] >= 0,
    "No negative incoming stock": procurement_math["incoming_stock_quantity"] >= 0,
}
for check, result in checks.items():
    print(f"{check}: {int(result.sum())} / {len(result)}")

discrepancies = procurement_math[
    ~checks["Projected stock matches"]
    | ~checks["Target stock matches"]
    | ~checks["Recommended quantity matches"]
]
print("\nRECORDS WITH CALCULATION DISCREPANCIES")
print(discrepancies[[
    "medicine_id", "forecast_demand_quantity", "lead_time_days", "safety_stock_days",
    "eligible_stock_quantity", "incoming_stock_quantity", "projected_stock_position",
    "calculated_projected_stock", "target_stock_quantity", "calculated_target_stock",
    "recommended_quantity", "calculated_recommended_quantity",
]].head(15).to_string(index=False))

group = "Procurement formula reconciliation"
for check, result in checks.items():
    formula_check = check in ("Projected stock matches", "Target stock matches", "Recommended quantity matches")
    record("8", f"{check}: records failing", (~result).sum(), 0, group=group,
           key="Procurement formula discrepancies" if formula_check else None)

NOVEMBER PROCUREMENT FORMULA RECONCILIATION
Projected stock matches: 100 / 100
Target stock matches: 100 / 100
Recommended quantity matches: 100 / 100
No negative recommended quantities: 100 / 100
No negative incoming stock: 100 / 100

RECORDS WITH CALCULATION DISCREPANCIES
Empty DataFrame
Columns: [medicine_id, forecast_demand_quantity, lead_time_days, safety_stock_days, eligible_stock_quantity, incoming_stock_quantity, projected_stock_position, calculated_projected_stock, target_stock_quantity, calculated_target_stock, recommended_quantity, calculated_recommended_quantity]
Index: []


In [20]:
# 8.3  Purchase order → procurement recommendation reconciliation
purchase_orders = datasets["purchase_orders"]
required_po_columns = ["purchase_order_id", "recommendation_id", "medicine_id", "status"]
missing_columns = [column for column in required_po_columns if column not in purchase_orders.columns]

print("PURCHASE ORDER–RECOMMENDATION RECONCILIATION")
print("=" * 80)
print("Purchase orders:", len(purchase_orders))
print("Procurement recommendations:", len(procurement_recommendations))
assert not missing_columns, f"Missing required PO columns: {missing_columns}"

po_reconciliation = purchase_orders.merge(
    procurement_recommendations[["recommendation_id", "medicine_id", "recommended_quantity"]],
    on="recommendation_id", how="left", validate="one_to_one",
    indicator=True, suffixes=("_po", "_recommendation"),
)
matched = po_reconciliation["_merge"].eq("both")

print("\nLINKAGE CHECKS")
print("Matched recommendations:", int(matched.sum()))
print("Unmatched purchase orders:", int((~matched).sum()))
print("Duplicate PO identifiers:", int(purchase_orders["purchase_order_id"].duplicated().sum()))
print("Duplicate recommendation links:", int(purchase_orders["recommendation_id"].duplicated().sum()))
print("Recommendations without a purchase order:",
      len(procurement_recommendations) - int(matched.sum()))

medicine_match = po_reconciliation["medicine_id_po"] == po_reconciliation["medicine_id_recommendation"]
medicine_discrepancies = int((~medicine_match & matched).sum())
print("\nMEDICINE CONSISTENCY")
print("Matching medicine references:", int(medicine_match.sum()))
print("Medicine reference discrepancies:", medicine_discrepancies)

quantity_candidates = ["quantity_ordered", "ordered_quantity", "order_quantity", "quantity"]
quantity_column = next((col for col in quantity_candidates if col in purchase_orders.columns), None)
assert quantity_column, "No recognised ordered-quantity column found."

quantity_match = po_reconciliation[quantity_column] == po_reconciliation["recommended_quantity"]
quantity_discrepancies = int((~quantity_match & matched).sum())
print("\nQUANTITY CONSISTENCY")
print("Ordered quantity column:", quantity_column)
print("Matching quantities:", int(quantity_match.sum()))
print("Quantity discrepancies:", quantity_discrepancies)

print("\nPURCHASE ORDER STATUS PROFILE")
print(purchase_orders["status"].value_counts().to_string())

group = "PO–recommendation reconciliation"
record("8", "Purchase orders not matched to a recommendation", (~matched).sum(), 0, group=group,
       key="PO recommendation linkage discrepancies")
record("8", "Duplicate purchase order identifiers",
       purchase_orders["purchase_order_id"].duplicated().sum(), 0, group=group,
       key="PO recommendation linkage discrepancies")
record("8", "Recommendations linked to more than one purchase order",
       purchase_orders["recommendation_id"].duplicated().sum(), 0, group=group,
       key="PO recommendation linkage discrepancies")
record("8", "Purchase order and recommendation medicines that differ", medicine_discrepancies, 0, group=group)
record("8", "Purchase order quantities that differ from the recommended quantity",
       quantity_discrepancies, 0, group=group, key="PO ordered quantity discrepancies")

display(section_table("8"))

PURCHASE ORDER–RECOMMENDATION RECONCILIATION
Purchase orders: 2455
Procurement recommendations: 2500

LINKAGE CHECKS
Matched recommendations: 2455
Unmatched purchase orders: 0
Duplicate PO identifiers: 0
Duplicate recommendation links: 0
Recommendations without a purchase order: 45

MEDICINE CONSISTENCY
Matching medicine references: 2455
Medicine reference discrepancies: 0

QUANTITY CONSISTENCY
Ordered quantity column: quantity_ordered
Matching quantities: 2455
Quantity discrepancies: 0

PURCHASE ORDER STATUS PROFILE
status
Placed with Supplier    2410
Pending Admin Review      26
Approved                  12
Rejected                   7

SECTION 8 CONFIRMATION: 14/14 checks passed


,Check,Observed,Expected,Status
0,Recommendations checked,100,100,PASS
1,Medicines whose recommendation stock differs f...,0,0,PASS
2,Recommendations whose forecast reference is no...,0,0,PASS
3,Recommendations whose demand differs from the ...,0,0,PASS
4,Projected stock matches: records failing,0,0,PASS
5,Target stock matches: records failing,0,0,PASS
6,Recommended quantity matches: records failing,0,0,PASS
7,No negative recommended quantities: records fa...,0,0,PASS
8,No negative incoming stock: records failing,0,0,PASS
9,Purchase orders not matched to a recommendation,0,0,PASS


### Findings: forecasts, recommendations and purchase orders
- **Eligible stock matches.** The 100 November recommendations cover 100 medicines, and for all 100 the stored FEFO-eligible stock equals the stock recalculated from the batch file (Active, in stock, valid through 22 November 2026).
- **Forecasts are synchronised.** All 100 recommendations reference the November forecast of the same medicine and carry the same forecast demand.
- **The procurement formula is correct.** Projected stock, target stock and recommended quantity are reproduced exactly for all 100 recommendations. None is negative.
- **Purchase orders trace to recommendations.** All 2,455 purchase orders link one-to-one to a recommendation with the same medicine and the same quantity, with no duplicates.
- **Note on reproducibility:** in the earlier version of this notebook, the first cell of this section relied on two variables (`procurement_recommendations`, `eligible_stock_by_medicine`) that no cell defined. It could only run because they were left over in the kernel. They are now defined explicitly, using the generator's eligibility rule, and the result is the same (100 of 100 match).

---
## 9. Acceptance report
**Purpose:** consolidate everything above into one decision.

**Decision rule:** the dataset is **accepted with documented limitations** only if every logged check passed. Otherwise it is **not accepted** and the failing checks must be investigated. The report below is computed from the check registry, not typed in:
- *Structural profiling* = Sections 0 and 1
- *Data integrity* = Sections 2 to 6
- *Data synchronization* = Section 7
- *Integration reconciliation* = Section 8

The limitations are not failures: they describe what the synthetic data cannot show.

In [21]:
# 9.1  Acceptance assessment (computed from the check registry)
results_table = pd.DataFrame(RESULTS)

section_titles = {
    "0": "Record counts",
    "1": "Structure and completeness",
    "2": "Primary keys and foreign keys",
    "3": "Dates and quantities",
    "4": "FEFO compliance",
    "5": "Dispensing overrides",
    "6": "Supplier master data",
    "7": "Procurement and receiving",
    "8": "Forecasts, recommendations and purchase orders",
}

overview = results_table.groupby("Section").agg(
    Checks=("Check", "count"),
    Passed=("Status", lambda s: int((s == "PASS").sum())),
)
overview["Failed"] = overview["Checks"] - overview["Passed"]
overview.insert(0, "Area", overview.index.map(section_titles))
overview.loc["Total"] = ["All sections", overview["Checks"].sum(),
                         overview["Passed"].sum(), overview["Failed"].sum()]
display(overview)

failed = results_table[results_table["Status"] == "FAIL"]
print("Failed checks:", len(failed))
if len(failed):
    display(failed[["Section", "Check", "Observed", "Expected"]])


def all_pass(rows):
    return len(rows) > 0 and all(row["Status"] == "PASS" for row in rows)


def rows_of(**conditions):
    return [row for row in RESULTS if all(row[k] == v for k, v in conditions.items())]


# Record-count results
count_pass = sum(
    len(datasets[name]) == expected for name, expected in expected_counts.items()
)

# The eight integration checks
integration_groups = [
    "PO–goods receipt reconciliation",
    "Goods receipt–batch reconciliation",
    "Opening inventory reconciliation",
    "Batch stock ledger reconciliation",
    "FEFO stock–procurement reconciliation",
    "Forecast–recommendation synchronization",
    "Procurement formula reconciliation",
    "PO–recommendation reconciliation",
]
integration_status = {g: all_pass(rows_of(Group=g)) for g in integration_groups}

# The nine key verified results (sum of the violations behind each label)
key_labels = [
    "Foreign-key violations",
    "FEFO eligibility violations",
    "Batch stock discrepancies",
    "Goods receipt–batch quantity discrepancies",
    "November FEFO stock discrepancies",
    "Forecast demand discrepancies",
    "Procurement formula discrepancies",
    "PO recommendation linkage discrepancies",
    "PO ordered quantity discrepancies",
]
key_results = {label: sum(row["Observed"] for row in rows_of(Key=label)) for label in key_labels}
key_evidence = {label: len(rows_of(Key=label)) for label in key_labels}

# Stage results
def section_pass(*sections):
    return all_pass([row for row in RESULTS if row["Section"] in sections])

stage_status = {
    "Structural profiling": section_pass("0", "1"),
    "Data integrity": section_pass("2", "3", "4", "5", "6"),
    "Data synchronization": section_pass("7"),
    "Integration reconciliation": section_pass("8"),
}
accepted = all(stage_status.values()) and all(integration_status.values())

# Documented limitations, with the numbers taken from this run
lead_words = {7: "seven"}
lead_text = lead_words.get(LIMITS["lead_time_days"], str(LIMITS["lead_time_days"]))
limitations = [
    f"{LIMITS['no_forecast_id']:,} historical procurement recommendations have no forecast_id.",
    f"{LIMITS['opening_inventory_batches']} opening inventory batches predate the goods receipt dataset;\n"
    f"   all are depleted.",
    f"{LIMITS['po_without_receipt']} purchase orders have no goods receipt:\n"
    f"   {LIMITS['po_not_placed']} are not placed with suppliers and {LIMITS['po_outstanding']} are outstanding deliveries\n"
    f"   expected after the snapshot.",
    "Supplier phone numbers may be inferred as numeric by pandas;\n   read them using string dtype.",
    f"All synthetic suppliers have a {lead_text}-day lead time and fictional\n   generic addresses.",
    "Synthetic FEFO override reasons represent test scenarios,\n   not real hospital documentation.",
]


def label(flag):
    return "PASS" if flag else "FAIL"


count_lines = "\n".join(
    f"{(name + ':'):<33}{len(datasets[name]):>5}" for name in expected_counts
)
integration_lines = "\n".join(f"{g}: {label(ok)}" for g, ok in integration_status.items())
key_lines = "\n".join(f"{k}: {v}" for k, v in key_results.items())
limitation_lines = "\n".join(f"{i}. {text}" for i, text in enumerate(limitations, 1))
stage_lines = "\n".join(f"{stage}: {label(ok)}" for stage, ok in stage_status.items())
classification = "ACCEPT WITH DOCUMENTED LIMITATIONS" if accepted else "NOT ACCEPTED: INVESTIGATION REQUIRED"

report_text = f'''
MEDICINE EXPIRY MANAGEMENT SYSTEM
FINAL SYNTHETIC DATASET ACCEPTANCE REPORT
============================================================

ASSESSMENT: {classification}

1. DATASET RECORD COUNT VERIFICATION

{count_lines}

Result: {count_pass}/{len(expected_counts)} datasets passed.

2. DATA SYNCHRONIZATION AND INTEGRATION

{integration_lines}

Result: {sum(integration_status.values())}/{len(integration_status)} integration checks passed.

3. KEY VERIFIED RESULTS

{key_lines}

4. DOCUMENTED LIMITATIONS

{limitation_lines}

5. FINAL ACCEPTANCE DECISION

{stage_lines}

OVERALL CLASSIFICATION:
{classification}

DATASET IMPORT STATUS:
NOT YET APPROVED

============================================================
Report generated from the independent dataset profiling notebook.
'''.strip()

print()
print(report_text)

,Area,Checks,Passed,Failed
Section,,,,
0,Record counts,10,10,0
1,Structure and completeness,5,5,0
2,Primary keys and foreign keys,5,5,0
3,Dates and quantities,17,17,0
4,FEFO compliance,10,10,0
5,Dispensing overrides,4,4,0
6,Supplier master data,4,4,0
7,Procurement and receiving,19,19,0
8,"Forecasts, recommendations and purchase orders",14,14,0


Failed checks: 0

MEDICINE EXPIRY MANAGEMENT SYSTEM
FINAL SYNTHETIC DATASET ACCEPTANCE REPORT

ASSESSMENT: ACCEPT WITH DOCUMENTED LIMITATIONS

1. DATASET RECORD COUNT VERIFICATION

medicines:                         100
suppliers:                          15
medicine_batches:                 2500
dispensing_transactions:         42617
demand_forecasts_evaluation:       600
demand_forecasts_november:         100
procurement_recommendations:      2500
purchase_orders:                  2455
goods_receipts:                   2400
goods_receipt_items:              2400

Result: 10/10 datasets passed.

2. DATA SYNCHRONIZATION AND INTEGRATION

PO–goods receipt reconciliation: PASS
Goods receipt–batch reconciliation: PASS
Opening inventory reconciliation: PASS
Batch stock ledger reconciliation: PASS
FEFO stock–procurement reconciliation: PASS
Forecast–recommendation synchronization: PASS
Procurement formula reconciliation: PASS
PO–recommendation reconciliation: PASS

Result: 8/8 integration ch

In [22]:
# 9.2  Save the acceptance report
report_path = REPORT_DIR / "final_dataset_acceptance_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print("FINAL REPORT GENERATED")
print("=" * 60)
print("File:", report_path.name)
print("Location:", report_path.relative_to(PROJECT_ROOT))
print("File exists:", report_path.exists())
print("File size:", report_path.stat().st_size, "bytes")

FINAL REPORT GENERATED
File: final_dataset_acceptance_report.txt
Location: synthetic_dataset_package\reports\profiling\final_dataset_acceptance_report.txt
File exists: True
File size: 2472 bytes


---
## 10. Findings and decisions log
This section answers *how was the data checked and treated?* It records exactly what happened to the data, what was found and what was decided.

### 10.1 What was done to the data
**Nothing was changed.** No record was modified, removed, filled in or recoded, and no CSV file was rewritten. The only handling is at **read time**, in memory:

| Handling | Where | Why |
|---|---|---|
| Files read with `encoding="utf-8-sig"` | 0.2 | removes a byte-order mark if present, so the first column name is clean |
| Default type detection for the profile; `suppliers.csv` read again with `phone_number` as text | 1.1, 6.1 | the default load reads `+2547…` as an integer, which would drop the `+` sign |
| Dates parsed into **copies** with `errors="coerce"` | 3, 4, 5 | an unreadable date is counted as invalid and never dropped |
| Empty text found with `strip() == ""` | 1.2, 6.1 | catches blank strings that a null test misses |
| Every merge uses `validate=` (`one_to_one`, `many_to_one`) | 7, 8 | a duplicated key would raise an error and not silently multiply rows |

### 10.2 Findings and decisions
| # | Finding | Section | Cause | Decision | Action on the data |
|---|---|---|---|---|---|
| 1 | `suppliers.phone_number` loads as an integer | 1, 6 | pandas reads `+2547…` as a number | Import as text; values are 13 characters and fit the 15-character model field | None. Read as text. |
| 2 | November forecasts have no `actual_demand_quantity` or `absolute_error` (100 rows) | 1 | November has not happened, so the forecasts are not yet evaluated | Expected | None |
| 3 | 2,400 recommendations have no `forecast_id` | 1 | Historical recommendations (Oct 2024 to Sep 2026) are not tied to a forecast by the generator's design; the application field is nullable | Accepted and documented | None |
| 4 | 100 batches have no goods receipt item | 7 | Opening inventory: `INITIAL` batch numbers, received 15 Oct 2024, all depleted | Accepted and documented | None |
| 5 | 55 purchase orders have no goods receipt | 7 | 45 are not yet placed with the supplier (26 Pending Admin Review, 12 Approved, 7 Rejected); 10 are placed and due on 16 Nov 2026 | Accepted. Not received stock. | None |
| 6 | 203 dispensing overrides depart from FEFO order | 4, 5 | The generator picks the second eligible batch at random (0.5%) | Accepted as documented exceptions: every one has a reason and is prescription-safe | None |
| 7 | 4 of the 203 overrides had too little earlier-expiring stock to cover the quantity | 5 | Overrides are random in the generator and do not depend on stock | **Observation only.** Not listed in the acceptance report; add it there if you want it formally documented. | None |
| 8 | All 15 suppliers share a 7-day lead time and one address | 6 | Generator constants | Documented limitation | None |
| 9 | Override reasons are synthetic test text | 5 | Generated by the script | Documented limitation | None |

Apart from these nine documented observations, **no check failed and nothing required correction** (see the overview table in 9.1).

### 10.3 Decision
- **Classification:** accept with documented limitations. The decision is computed in 9.1.
- **Import status: not yet approved.** The methodology (section 12) forbids importing the data into the Django database until the whole package has been reviewed and explicitly approved. This notebook gives the evidence for that review. The approval itself is a separate decision.
- **For the import step:** read `phone_number` as text, keep `forecast_id` empty where it is empty, and do not count the 10 outstanding purchase orders or the 100 depleted opening batches as received stock.

---
## Appendix A: generator rules referred to in this review
Several findings depend on how the generator creates the data. Instead of printing hundreds of lines of source, this cell extracts only the rules the review relies on (constants, the batch eligibility rules, the override rule, the supplier phone format and the CSV writer) directly from `scripts/generation_script.py`, so they always match the script actually used. This is read-only.

In [23]:
# A.1  Extract the generator rules used in this review
import ast
import re

generator_path = PACKAGE_DIR / "scripts" / "generation_script.py"

if not generator_path.is_file():
    print("Generator script not found:", generator_path.relative_to(PROJECT_ROOT))
else:
    source = generator_path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)
    functions = {node.name: node for node in tree.body if isinstance(node, ast.FunctionDef)}

    def show(title, first_line, last_line):
        print(f"\n{'=' * 78}\n{title}  (lines {first_line}–{last_line})\n{'=' * 78}")
        for number in range(first_line, last_line + 1):
            print(f"{number:5}: {lines[number - 1]}")

    # Constants
    wanted = ("PERIOD_DAYS", "HISTORY_PERIODS", "MAX_PRESCRIPTION_DAYS",
              "MEDICINE_COUNT", "SUPPLIER_COUNT", "OVERRIDE_PROBABILITY")
    print("CONSTANTS")
    for number, line in enumerate(lines, 1):
        if re.match(rf"({'|'.join(wanted)})\s*=", line):
            print(f"{number:5}: {line}")

    # Batch eligibility and FEFO ordering
    for name in ("batch_is_eligible", "eligible_batches"):
        show(f"{name}()", functions[name].lineno, functions[name].end_lineno)

    # Override rule inside allocate_fefo
    allocate = functions["allocate_fefo"]
    hits = [n for n in range(allocate.lineno, allocate.end_lineno + 1) if "use_override" in lines[n - 1]]
    show("allocate_fefo(): override rule", max(hits[0] - 2, allocate.lineno), min(hits[0] + 19, allocate.end_lineno))

    # Supplier phone format and fixed values
    suppliers_fn = functions["generate_suppliers"]
    show("generate_suppliers(): phone, lead time and address", suppliers_fn.lineno, suppliers_fn.end_lineno)

    # CSV export
    write_csv = functions["write_csv"]
    show("write_csv()", write_csv.lineno, write_csv.end_lineno)

CONSTANTS
   78: PERIOD_DAYS = 30
   79: HISTORY_PERIODS = 3
   80: MAX_PRESCRIPTION_DAYS = 14
   82: MEDICINE_COUNT = 100
   83: SUPPLIER_COUNT = 15
   85: OVERRIDE_PROBABILITY = 0.005

batch_is_eligible()  (lines 392–411)
  392: def batch_is_eligible(
  393:     batch: dict,
  394:     medicine_id: str,
  395:     required_valid_until: date
  396: ) -> bool:
  397:     """
  398:     A batch is eligible only when:
  399:     - It belongs to the requested medicine.
  400:     - It is Active.
  401:     - It has positive available stock.
  402:     - Its expiry date covers the required validity date.
  403:     """
  404: 
  405:     return (
  406:         batch["medicine_id"] == medicine_id
  407:         and batch["status"] == "Active"
  408:         and int(batch["quantity_available"]) > 0
  409:         and date.fromisoformat(batch["expiry_date"])
  410:             >= required_valid_until
  411:     )

eligible_batches()  (lines 430–462)
  430: def eligible_batches(
  431:     ba